# Milestone 1 — Project Setup & Dataset Preparation

**Dataset:** COCO 2017 Captions (`val2017`) · **Generator:** Stable Diffusion v1.5 (`diffusers`) ·
**Text encoders:** CLIP and BERT (`transformers`)

This notebook is the **narrative** of milestone 1: it defines the steps and shows the results. The
reusable code lives next door in `src/`, so a later milestone can import a stage instead of copying
a cell out of here.

| Where | What is in it |
|---|---|
| `src/config.py` | every tunable value and every path — the only file to edit to change a run |
| `src/utils.py` | seeding, and saving figures and tables to `outputs/` |
| `src/modules/data_preprocessing.py` | caption cleaning, subset construction, image preprocessing |
| `src/modules/clip_embeddings.py` | CLIP text and image encoders |
| `src/modules/bert_embeddings.py` | BERT text encoder |
| `src/modules/storage.py` | the manifest, the `.npy` arrays, metadata, cache logic |
| `src/modules/sanity_checks.py` | shape health, retrieval, sibling agreement, neighbours |

| # | Section | What happens here |
|---|---------|-------------------|
| 1 | Environment Setup | verify the toolchain `pyproject.toml` installed |
| 2 | Configuration, Paths & Reproducibility | load `config.py`, fix the seed, check the inputs |
| 3 | Dataset Loading | COCO 2017 paired image–caption data |
| 4 | Caption Cleaning | `data_preprocessing.clean_captions` + `filter_captions` |
| 5 | Subset Construction | `data_preprocessing.build_subset` — 3,000 × 2 = 6,000 pairs |
| 6 | Image Preprocessing | `data_preprocessing.preprocess_image` — 512×512 RGB |
| 7 | Text Embeddings — CLIP | `clip_embeddings` |
| 8 | Text Embeddings — BERT | `bert_embeddings` |
| 9 | Storage | `storage` — embeddings persisted alongside their pairs |
| 10 | Sanity Checks | `sanity_checks` |
| 11 | Prompt → Image Test Runs | 5 prompts through Stable Diffusion v1.5 |
| 12 | Artefacts | what this notebook leaves on disk |

**Inputs** (already on disk, nothing is downloaded for the dataset)

- `data/annotations/captions_val2017.json` — COCO 2017 validation captions
- `data/val2017/*.jpg` — the 5,000 images those captions describe

**Outputs**

- `data/processed/pairs.parquet` + `pairs.csv` — the curated manifest
- `data/processed/images_512/` — the 512×512 RGB image cache
- `data/processed/embeddings/*.npy` — CLIP and BERT text embeddings, row-aligned to the manifest
- `data/processed/metadata.json` — provenance for everything above
- `outputs/generations/*.png` — the 5 test-run images


---
## 0. Setup - this notebook is self-contained

Everything it needs is **in this file**. There is no repository to clone and no `src/` to import:
the cells below write the project's modules to disk beside the notebook, then import them
normally. The code is byte-identical to the tested version in the repo.

Writing real files rather than pasting the functions into the notebook's own namespace is
deliberate. `clip_embeddings`, `bert_embeddings` and `gpt2_embeddings` each define `load`,
`encode_text` and `describe`, so flattening them together would silently shadow two of the three.

**Paths.** `config.py` anchors everything to the working directory, so artefacts land in `./data/`
and `./outputs/` - `/kaggle/working/...` on Kaggle, which is writable. Milestone 2 additionally
needs milestone 1's `data/processed/`: attach it as a Dataset and the next cell finds it, or run
milestone 1 first in the same session.

In [ ]:
# Install only what is actually missing -- Kaggle ships torch and transformers already.
import importlib.util
import subprocess
import sys
from pathlib import Path

missing = [p for p in ("diffusers", "transformers", "accelerate", "safetensors", "ftfy", "scipy")
           if importlib.util.find_spec(p) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

Path("modules").mkdir(exist_ok=True)
print("ready -- writing modules next")

In [ ]:
# Point config.py at milestone 1's data if it arrived as an attached Dataset.
# No-op everywhere else, including a local run.
import os
import shutil
from pathlib import Path

if Path("/kaggle/input").is_dir():
    attached = sorted(p for p in Path("/kaggle/input").iterdir() if p.is_dir())
    source = next((d for d in attached if (d / "processed" / "pairs.parquet").is_file()), None)
    if source is not None:
        work = Path("/kaggle/working/data")
        if not (work / "processed").is_dir():
            print(f"copying {source.name}/processed -> {work} ...")
            shutil.copytree(source / "processed", work / "processed")
        os.environ["T2I_DATA_DIR"] = str(work)
        print(f"using milestone 1 data from dataset '{source.name}'")
    else:
        print("no milestone 1 dataset attached -- run milestone 1 in this session first.")
        print("attached:", [d.name for d in attached] or "nothing")
else:
    print("not on Kaggle -- using ./data and ./outputs")

In [ ]:
%%writefile config.py
"""Every tunable value and every path for the milestone-1 pipeline.

Nothing in this file does any work. It is the one place to change the scale of a
run, the models, the caption filters or where artefacts land -- edit a value
here, restart the kernel, re-run the notebook, and nothing else needs touching.

Paths can also be overridden with environment variables, which is how this runs
on Kaggle without editing the file:

    T2I_PROJECT_ROOT   repo root            (default: the parent of src/)
    T2I_DATA_DIR       dataset + artefacts  (default: <root>/data)
    T2I_OUTPUT_DIR     plots, tables, PNGs  (default: <root>/outputs)
"""

import os
from pathlib import Path

import torch

# ---- Paths -------------------------------------------------------------
# Derived from this file's own location, so it does not matter where the kernel
# was started from -- src/config.py -> parents[1] is the repo root.
PROJECT_ROOT = Path(os.environ.get("T2I_PROJECT_ROOT") or Path.cwd())

DATA_DIR = Path(os.environ.get("T2I_DATA_DIR") or PROJECT_ROOT / "data")
ANN_FILE = DATA_DIR / "annotations" / "captions_val2017.json"
IMAGES_DIR = DATA_DIR / "val2017"

PROCESSED_DIR = DATA_DIR / "processed"
IMAGES_512_DIR = PROCESSED_DIR / "images_512"      # 512x512 RGB JPEG cache
EMB_DIR = PROCESSED_DIR / "embeddings"             # the .npy arrays
MANIFEST_PARQUET = PROCESSED_DIR / "pairs.parquet"
MANIFEST_CSV = PROCESSED_DIR / "pairs.csv"
METADATA_FILE = PROCESSED_DIR / "metadata.json"

OUTPUT_DIR = Path(os.environ.get("T2I_OUTPUT_DIR") or PROJECT_ROOT / "outputs")
GENERATIONS_DIR = OUTPUT_DIR / "generations"       # generated PNGs
PLOTS_DIR = OUTPUT_DIR / "plots"                   # every figure, as PNG
TABLES_DIR = OUTPUT_DIR / "tables"                 # every summary table, as JSON

SWEEP_DIR = OUTPUT_DIR / "sweep"                   # one sub-directory per swept config

ALL_DIRS = (PROCESSED_DIR, IMAGES_512_DIR, EMB_DIR, GENERATIONS_DIR, PLOTS_DIR, TABLES_DIR,
            SWEEP_DIR)


def ensure_dirs():
    """Create every output directory. Called once near the top of the notebook."""
    for d in ALL_DIRS:
        d.mkdir(parents=True, exist_ok=True)


# ---- Subset ------------------------------------------------------------
N_IMAGES = 3000             # distinct images kept
CAPTIONS_PER_IMAGE = 2      # captions kept per image -> 3000 x 2 = 6000 pairs
IMAGE_SIZE = 512            # Stable Diffusion v1.x native resolution
SEED = 42                   # single source of randomness for the whole pipeline

# ---- Caption filters ---------------------------------------------------
MIN_WORDS, MAX_WORDS = 3, 60
MIN_CHARS, MAX_CHARS = 10, 250

# ---- Models ------------------------------------------------------------
CLIP_JOINT_MODEL = "openai/clip-vit-base-patch32"    # text + vision towers -> shared 512-d space
CLIP_SD_MODEL = "openai/clip-vit-large-patch14"      # the text encoder SD v1.5 is conditioned on
BERT_MODEL = "google-bert/bert-base-uncased"         # text-only Transformer, for comparison
GPT2_MODEL = "openai-community/gpt2"                 # autoregressive encoder, milestone 2's optional arm
SD_REPO_CANDIDATES = [                               # tried in order; the original runwayml repo is gone
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    "sd-legacy/stable-diffusion-v1-5",
    "CompVis/stable-diffusion-v1-4",
]

CLIP_MAX_TOKENS = 77   # fixed by CLIP's positional embedding table
BERT_MAX_TOKENS = 64   # COCO captions are short; 64 covers them with room to spare
GPT2_MAX_TOKENS = 64   # matched to BERT so the encoder comparison varies only the model
BATCH_SIZE = 64

# ---- Generation (section 11; the knobs milestone 2 sweeps) -------------
N_STEPS = 30           # denoising steps
GUIDANCE = 7.5         # classifier-free guidance scale
N_PROMPTS = 5          # test prompts drawn from the curated subset
PROMPT_MIN_WORDS = 8   # prompts long enough to describe a scene, not one object
PROMPT_MAX_WORDS = 18

# ---- Precision ---------------------------------------------------------
# Text encoders in float32: they are small, and float32 keeps the stored embeddings
# free of rounding so cosine similarities are exact. Stable Diffusion in float16 on
# GPU: the UNet is the expensive part and fp16 roughly halves VRAM and step time.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENCODER_DTYPE = torch.float32
SD_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

# ---- Toggles -----------------------------------------------------------
# The 77x768 sequence tensors are ~710 MB at 6,000 captions. The SD text encoder is
# deterministic in eval mode, so re-running it reproduces them bit-for-bit in well
# under a minute -- cheaper to recompute than to store.
SAVE_SEQ_EMBEDDINGS = False
RUN_SD_GENERATION = True    # section 11 needs ~4 GB of Stable Diffusion weights on first run
N_ALIGN_IMAGES = 500        # images put through CLIP's vision tower for the retrieval check
REUSE_EXISTING = True       # reuse cached embeddings when they match this run's subset
FORCE_RECOMPUTE = False     # set True to ignore every cache and redo all work


# ---- Milestone 2: the sweep ---------------------------------------------
N_FIXED_PROMPTS = 10   # every config generates this same set, from the same latents
LATENT_SEED = SEED     # latents are built once, on CPU, and passed into every run

CFG_GRID = (1.0, 4.0, 7.5, 11.0, 15.0)   # spec sweeps 1-15; 7.5 is the SD 1.5 default
GUIDANCE_RESCALE = 0.7                   # corrects the oversaturation CFG > 12 causes

# Sampler -> step count. The counts differ on purpose: DPMSolver++ and UniPC reach at
# 20 steps what PNDM needs 50 for, so an equal-step comparison would measure nothing
# but the step count.
SAMPLERS = {
    "pndm": 50,      # SD 1.5 default -- the baseline
    "ddim": 50,      # deterministic at eta=0
    "lms": 30,       # multistep; needs scipy
    "euler": 30,
    "euler_a": 30,   # ancestral -- adds noise each step, never converges
    "dpmpp": 20,
    "unipc": 20,
}

# Noise-schedule axes. Swept as a product against one fixed sampler, so the only
# variable is the schedule.
# SD 1.5's shipped scheduler config omits timestep_spacing, so each scheduler class would
# fall back to its own default -- PNDM and DDIM to "leading", the rest to "linspace". The
# guidance and sampler grids pin it so that axis stays still while the schedule grid sweeps it.
BASE_TIMESTEP_SPACING = "leading"   # SD legacy, what v1.5 was trained under

TIMESTEP_SPACINGS = ("leading", "trailing", "linspace")
ZERO_SNR_OPTIONS = (False, True)
KARRAS_OPTIONS = (False, True)
SCHEDULE_SAMPLER = "dpmpp"

DDPM_STEPS = 1000      # the illustration, not a sweep arm
DDPM_N_PROMPTS = 2

# Batch 4, not the 8 an unshared 16 GB T4 allows: this targets an 8 GB RTX 5060 Laptop
# GPU that is also driving the display. SD 1.5 in fp16 is ~2.5 GB of weights and roughly
# 0.6 GB of activations per 512x512 image.
GEN_BATCH_SIZE = 4

In [ ]:
%%writefile utils.py
"""Small helpers every section uses: seeding, path printing, and saving figures
and summary tables to `outputs/`.

Kept out of `config.py` on purpose -- config holds values you edit, this holds
code you call.
"""

import json
import os
import random
from pathlib import Path

import numpy as np
import torch

import config as cfg


def set_seed(seed=None):
    """Reseed every RNG the pipeline touches.

    Called once at the start and again before generation, so the subset, the
    sampled examples and the generated images are identical on every re-run.
    """
    seed = cfg.SEED if seed is None else seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Deterministic convolution kernels: slightly slower, but two runs produce
    # identical tensors.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return seed


def rel(path):
    """`path` relative to the project root, as a string, for printing.

    Falls back to a computed relative path instead of raising when the path sits
    outside the root -- which happens on Kaggle, where the data and the code live
    on different mounts -- and to the absolute path when even that is impossible,
    as on Windows when the two are on different drives.
    """
    p = Path(path)
    try:
        return p.relative_to(cfg.PROJECT_ROOT).as_posix()
    except ValueError:
        pass
    try:
        return Path(os.path.relpath(p, cfg.PROJECT_ROOT)).as_posix()
    except ValueError:
        return p.as_posix()


def save_fig(fig, name):
    """Write a figure to `outputs/plots/` and return its path for printing."""
    cfg.PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    path = cfg.PLOTS_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    return rel(path)


def save_table(name, data):
    """Write a summary table to `outputs/tables/` as JSON.

    `default=float` converts numpy scalars, which json cannot serialise.
    """
    cfg.TABLES_DIR.mkdir(parents=True, exist_ok=True)
    path = cfg.TABLES_DIR / f"{name}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=float)
    return rel(path)

In [ ]:
%%writefile modules/__init__.py
"""Pipeline stages, written by the setup cells above."""

In [ ]:
%%writefile modules/bert_embeddings.py
"""BERT text encoder -- the comparison encoder.

`bert-base-uncased` is the same architecture class and roughly the same size as
CLIP's text tower, but trained with masked language modelling on text alone. It
has never seen an image. That difference is the point: both produce 768-d
sentence vectors that look equally reasonable on paper, so the only way to tell
them apart is to measure them, which is what `sanity_checks` does.

Pooling: BERT ships a `pooler_output`, but it is the tanh-projected [CLS] state
trained for next-sentence prediction, not for sentence similarity -- using it as
a sentence embedding is a well-documented trap. We compute two honest
alternatives instead, and prefer mean pooling when they disagree.

Swapping in a different text encoder means writing a file like this one with the
same two functions and changing one call in the notebook.
"""

import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

import config as cfg


def load(model_name=None, device=None, dtype=None):
    """Load BERT. Returns `(tokenizer, model)`."""
    model_name = cfg.BERT_MODEL if model_name is None else model_name
    device = cfg.DEVICE if device is None else device
    dtype = cfg.ENCODER_DTYPE if dtype is None else dtype

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device=device, dtype=dtype).eval()
    return tokenizer, model


def mean_pool(last_hidden_state, attention_mask):
    """Mask-aware mean over the token dimension, so padding contributes nothing."""
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    return (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


@torch.no_grad()
def encode_text(texts, tokenizer, model, batch_size=None, max_tokens=None, device=None):
    """Encode captions two ways. Returns `(cls, mean)`, each (N, 768) float32.

    `cls`  -- the first token's final hidden state, raw.
    `mean` -- the attention-mask-weighted mean over all real tokens, the standard
              recipe for sentence similarity from an MLM encoder.
    """
    batch_size = cfg.BATCH_SIZE if batch_size is None else batch_size
    max_tokens = cfg.BERT_MAX_TOKENS if max_tokens is None else max_tokens
    device = cfg.DEVICE if device is None else device

    cls_out, mean_out = [], []
    for i in tqdm(range(0, len(texts), batch_size), desc="BERT"):
        toks = tokenizer(
            texts[i:i + batch_size], padding="max_length", max_length=max_tokens,
            truncation=True, return_tensors="pt",
        ).to(device)
        hidden = model(**toks).last_hidden_state
        cls_out.append(hidden[:, 0].float().cpu())
        mean_out.append(mean_pool(hidden, toks["attention_mask"]).float().cpu())
    return torch.cat(cls_out).numpy(), torch.cat(mean_out).numpy()


def describe(model):
    """Parameter, layer and vocab counts, for the notebook to print after loading."""
    return {
        "hidden_size": model.config.hidden_size,
        "layers": model.config.num_hidden_layers,
        "vocab_size": model.config.vocab_size,
        "parameters_millions": sum(p.numel() for p in model.parameters()) / 1e6,
    }

In [ ]:
%%writefile modules/clip_embeddings.py
"""CLIP text and image encoders.

Two checkpoints, doing two different jobs:

  `clip-vit-base-patch32`   pooled 512-d vectors in a space shared by text AND
                            images, so a caption vector and a photo vector are
                            directly comparable. This is what makes the
                            text -> image retrieval check possible.
  `clip-vit-large-patch14`  the exact text encoder Stable Diffusion v1.5 is
                            conditioned on. Its 77 x 768 per-token states are the
                            same *kind* of tensor the UNet cross-attends to -- no
                            substitutes, because a pretrained UNet only understands
                            the space it was trained in.

**`encode_sequence` here is for analysis, not for conditioning.** It passes an
attention mask, which the pipeline deliberately does not; see the warning on that
function. To build `prompt_embeds`, use `conditioning.encode_prompts`.

The conditioning tensor is one vector per token, not one per sentence, because
the generator reads the prompt token by token. That is how "a *red* bus" colours
the bus instead of tinting the whole picture red.
"""

import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPImageProcessor, CLIPModel, CLIPTextModel, CLIPTokenizer

import config as cfg


def pooled(out):
    """Unwrap `get_text_features` / `get_image_features` across transformers versions.

    transformers >= 5 returns a pooling output object; earlier versions return a
    bare tensor. `pooler_output` is ALREADY projected into the joint space --
    applying `model.text_projection` on top of it would be wrong.
    """
    return out.pooler_output if hasattr(out, "pooler_output") else out


# ---------------------------------------------------------------------------
# Joint text + vision model
# ---------------------------------------------------------------------------
def load_joint(model_name=None, device=None, dtype=None):
    """Load the joint-space CLIP. Returns `(tokenizer, model, image_processor)`."""
    model_name = cfg.CLIP_JOINT_MODEL if model_name is None else model_name
    device = cfg.DEVICE if device is None else device
    dtype = cfg.ENCODER_DTYPE if dtype is None else dtype

    tokenizer = CLIPTokenizer.from_pretrained(model_name)
    model = CLIPModel.from_pretrained(model_name).to(device=device, dtype=dtype).eval()
    processor = CLIPImageProcessor.from_pretrained(model_name)
    return tokenizer, model, processor


def token_lengths(texts, tokenizer):
    """Untruncated token count per caption.

    CLIP's positional table stops at 77 tokens, so anything longer is silently
    truncated and the model never sees the end of the caption. Measuring this is
    how we know the word-count filter is already keeping us under the ceiling.
    """
    return np.array([len(tokenizer(t, truncation=False)["input_ids"]) for t in texts])


@torch.no_grad()
def encode_text(texts, tokenizer, model, batch_size=None, max_tokens=None, device=None):
    """Pooled 512-d vectors in CLIP's shared text/image space. Returns (N, 512) float32."""
    batch_size = cfg.BATCH_SIZE if batch_size is None else batch_size
    max_tokens = cfg.CLIP_MAX_TOKENS if max_tokens is None else max_tokens
    device = cfg.DEVICE if device is None else device

    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc="CLIP joint-space"):
        toks = tokenizer(
            texts[i:i + batch_size], padding="max_length", max_length=max_tokens,
            truncation=True, return_tensors="pt",
        ).to(device)
        out.append(pooled(model.get_text_features(**toks)).float().cpu())
    return torch.cat(out).numpy()


@torch.no_grad()
def encode_images(images, processor, model, batch_size=32, device=None, progress=True):
    """Pooled 512-d image vectors in the same space as `encode_text`.

    `images` may be paths or already-open PIL images.
    """
    device = cfg.DEVICE if device is None else device
    out = []
    steps = range(0, len(images), batch_size)
    for i in tqdm(steps, desc="CLIP image encode", disable=not progress):
        batch = [
            img if isinstance(img, Image.Image) else Image.open(img)
            for img in images[i:i + batch_size]
        ]
        batch = [img.convert("RGB") for img in batch]
        px = processor(images=batch, return_tensors="pt").to(device)
        out.append(pooled(model.get_image_features(**px)).float().cpu())
    return torch.cat(out).numpy()


# ---------------------------------------------------------------------------
# Stable Diffusion conditioning encoder
# ---------------------------------------------------------------------------
def load_sd_text_encoder(model_name=None, device=None, dtype=None):
    """Load the text encoder SD v1.5 is conditioned on. Returns `(tokenizer, model)`."""
    model_name = cfg.CLIP_SD_MODEL if model_name is None else model_name
    device = cfg.DEVICE if device is None else device
    dtype = cfg.ENCODER_DTYPE if dtype is None else dtype

    tokenizer = CLIPTokenizer.from_pretrained(model_name)
    model = CLIPTextModel.from_pretrained(model_name).to(device=device, dtype=dtype).eval()
    return tokenizer, model


@torch.no_grad()
def encode_sequence(texts, tokenizer, model, batch_size=None, max_tokens=None, device=None):
    """Per-token hidden states (N, 77, 768) plus the pooled EOS vector (N, 768).

    **Do not feed the sequence tensor to a pipeline as `prompt_embeds`.** This passes
    an attention mask; Stable Diffusion's own `encode_prompt` does not. CLIP's text
    tower is causal, so the mask leaves every real token bit-identical -- the pooled
    vector this returns is unaffected, which is why it is fine for the analysis it was
    written for -- but it moves every padded position by ~30 on a scale of 33, and the
    UNet cross-attends to all 77. Images come out different and measurably worse
    (CLIP score 0.23 against 0.34) with nothing raising an error.

    Use `conditioning.encode_prompts` for conditioning.

    The sequence tensor comes back as float16 -- it is ~710 MB at 6,000 captions
    in float32, and halving it costs nothing we can measure downstream.
    """
    batch_size = cfg.BATCH_SIZE if batch_size is None else batch_size
    max_tokens = cfg.CLIP_MAX_TOKENS if max_tokens is None else max_tokens
    device = cfg.DEVICE if device is None else device

    seq_out, pool_out = [], []
    for i in tqdm(range(0, len(texts), batch_size), desc="CLIP sequence"):
        toks = tokenizer(
            texts[i:i + batch_size], padding="max_length", max_length=max_tokens,
            truncation=True, return_tensors="pt",
        ).to(device)
        out = model(**toks)
        seq_out.append(out.last_hidden_state.to(torch.float16).cpu())
        pool_out.append(out.pooler_output.float().cpu())
    return torch.cat(seq_out).numpy(), torch.cat(pool_out).numpy()


def describe(model):
    """Parameter and layer counts, for the notebook to print after loading."""
    config = model.config
    out = {"parameters_millions": sum(p.numel() for p in model.parameters()) / 1e6}
    if hasattr(config, "text_config"):
        out["text_hidden_size"] = config.text_config.hidden_size
        out["vision_hidden_size"] = config.vision_config.hidden_size
        out["joint_projection_dim"] = config.projection_dim
    else:
        out["hidden_size"] = config.hidden_size
        out["layers"] = config.num_hidden_layers
    return out

In [ ]:
%%writefile modules/data_preprocessing.py
"""Caption cleaning, subset construction and image preprocessing.

Three stages, in the order the notebook runs them:

  1. `clean_captions` normalises raw COCO text; `filter_captions` drops the rows
     that are empty, duplicated, too short, too long, or whose image is missing.
  2. `build_subset` makes the deterministic N_IMAGES x CAPTIONS_PER_IMAGE
     selection and assigns `pair_idx`, the join key everything downstream uses.
  3. `preprocess_image` turns one photo into a 512x512 RGB image, with
     `resize_image` and `normalize_pixels` underneath it;
     `cache_resized_images` runs it over the whole subset and caches the result.
"""

import html
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
from ftfy import fix_text
from PIL import Image, ImageOps
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm

import config as cfg

# ---------------------------------------------------------------------------
# 1. Caption cleaning
# ---------------------------------------------------------------------------
# Regexes rather than hand-written character lists, because a whitelist removes
# every character we did NOT think of -- emoji, currency symbols, stray Unicode --
# whereas a blacklist only removes the ones someone remembered.
_HTML_TAG = re.compile(r"<[^>]+>")
_URL = re.compile(r"(?:https?://|www\.)\S+")
_SQUOTES = re.compile(r"[‘’‚‛′`]")          # curly single quotes -> '
_DQUOTES = re.compile(r"[“”„″]")                 # curly double quotes -> "
_DASHES = re.compile(r"[‐-―−]")                       # en / em / figure dashes -> -
_ALLOWED = re.compile(r"[^a-z0-9\s'.,!?-]")                          # whitelist: everything else -> space
_MULTI_PUNCT = re.compile(r"([.,!?-])\1+")                           # "!!!" -> "!"
_SPACE_PUNCT = re.compile(r"\s+([.,!?])")                            # "cat ." -> "cat."
_WHITESPACE = re.compile(r"\s+")                                     # any run of whitespace -> " "

CLEANING_STEPS = [
    "ftfy mojibake repair", "html unescape + tag strip", "url removal",
    "typographic folding (regex)", "unicode NFKD + ascii fold", "lowercase + strip",
    "whitelist regex", "punctuation collapse", "whitespace -> single space",
]


def clean_caption(text):
    """Normalise one raw COCO caption into clean, tokenizer-safe lowercase text.

    Order matters: each step assumes the previous one has run.
    """
    if not isinstance(text, str):
        return ""

    t = fix_text(text)                                   # 1. repair mojibake
    t = html.unescape(t)                                 # 2. &amp; -> &
    t = _HTML_TAG.sub(" ", t)                            #    strip HTML tags
    t = _URL.sub(" ", t)                                 # 3. drop URLs

    t = _SQUOTES.sub("'", t)                             # 4. typographic folding
    t = _DQUOTES.sub('"', t)
    t = _DASHES.sub("-", t)

    t = unicodedata.normalize("NFKD", t)                 # 5. decompose accents...
    t = t.encode("ascii", "ignore").decode("ascii")      #    ...and drop them

    t = t.lower().strip()                                # 6. lowercase + strip
    t = _ALLOWED.sub(" ", t)                             # 7. whitelist
    t = _MULTI_PUNCT.sub(r"\1", t)                       # 8. collapse repeated punctuation
    t = _SPACE_PUNCT.sub(r"\1", t)
    t = _WHITESPACE.sub(" ", t).strip()                  # 9. single spaces only

    return t


def clean_captions(df, source_col="caption_raw", progress=True):
    """Clean every caption in `df[source_col]`, adding `caption`, `n_words`, `n_chars`.

    Returns a new frame; the input is left alone.
    """
    out = df.copy()
    if progress:
        tqdm.pandas(desc="Cleaning captions")
        out["caption"] = out[source_col].progress_apply(clean_caption)
    else:
        out["caption"] = out[source_col].apply(clean_caption)
    out["n_words"] = out["caption"].str.split().str.len().fillna(0).astype(int)
    out["n_chars"] = out["caption"].str.len()
    return out


def cleaning_rule_hits(df, source_col="caption_raw"):
    """How much of the cleaner is load-bearing on this corpus, as opposed to defensive.

    COCO captions turn out to be plain ASCII already, so the Unicode steps change
    nothing here. We keep them anyway -- they cost nothing, and they are what makes
    quotes, dashes and accents come out consistent if any caption does carry them.
    """
    raw = df[source_col]
    return {
        "n_captions": int(len(df)),
        "altered_by_cleaning": int((df["caption"] != raw).sum()),
        "with_non_ascii": int(raw.map(lambda c: any(ord(ch) > 127 for ch in c)).sum()),
        "with_html": int(raw.str.contains(r"&\w+;|<[^>]+>", regex=True).sum()),
        "with_whitespace_to_fix": int(raw.str.contains(r"\s\s|^\s|\s$", regex=True).sum()),
        "needing_lowercase": int((raw != raw.str.lower()).sum()),
    }


def filter_captions(df, min_words=None, max_words=None, min_chars=None, max_chars=None):
    """Drop unusable rows, one rule at a time.

    Returns `(filtered_df, stages)` where `stages` is a list of
    `(rule name, rows remaining)` -- reporting it as a table rather than one
    chained expression makes it obvious if a single rule is dropping too much.
    """
    min_words = cfg.MIN_WORDS if min_words is None else min_words
    max_words = cfg.MAX_WORDS if max_words is None else max_words
    min_chars = cfg.MIN_CHARS if min_chars is None else min_chars
    max_chars = cfg.MAX_CHARS if max_chars is None else max_chars

    stages, out = [], df.copy()
    stages.append(("start", len(out)))

    out = out[out["caption"].str.strip().astype(bool)]
    stages.append(("drop empty", len(out)))

    out = out[(out["n_words"] >= min_words) & (out["n_chars"] >= min_chars)]
    stages.append((f"drop short (<{min_words}w / <{min_chars}c)", len(out)))

    out = out[(out["n_words"] <= max_words) & (out["n_chars"] <= max_chars)]
    stages.append((f"drop long (>{max_words}w / >{max_chars}c)", len(out)))

    out = out[out["image_path"].map(lambda p: Path(p).is_file())]
    stages.append(("drop missing image file", len(out)))

    out = out.drop_duplicates(subset="caption", keep="first")
    stages.append(("drop duplicate captions", len(out)))

    return out.reset_index(drop=True), stages


# ---------------------------------------------------------------------------
# 2. Subset construction
# ---------------------------------------------------------------------------
def build_subset(df, n_images=None, captions_per_image=None, seed=None):
    """Keep `n_images` distinct images with `captions_per_image` captions each.

    Deterministic: captions are ranked within each image by annotation id, images
    with enough surviving captions are eligible, and the images are drawn with a
    seeded generator. Re-running produces exactly the same rows.

    Every row gets `pair_idx` (0 .. N-1), which is the join key between the
    manifest and every embedding array -- row *i* of any .npy file is the caption
    on row *i* of the manifest, and nothing else needs to be tracked.

    Returns `(subset, info)`.
    """
    n_images = cfg.N_IMAGES if n_images is None else n_images
    captions_per_image = cfg.CAPTIONS_PER_IMAGE if captions_per_image is None else captions_per_image
    seed = cfg.SEED if seed is None else seed

    ranked = df.sort_values(["image_id", "ann_id"]).copy()
    ranked["cap_rank"] = ranked.groupby("image_id").cumcount()

    counts = ranked.groupby("image_id").size()
    eligible_ids = counts[counts >= captions_per_image].index

    rng = np.random.default_rng(seed)
    take = min(n_images, len(eligible_ids))
    chosen_ids = set(rng.choice(np.asarray(eligible_ids), size=take, replace=False))

    subset = ranked[ranked["image_id"].isin(chosen_ids) & (ranked["cap_rank"] < captions_per_image)]
    subset = subset.sort_values(["image_id", "cap_rank"]).reset_index(drop=True)
    subset.insert(0, "pair_idx", np.arange(len(subset)))

    assert subset["image_id"].nunique() == take
    assert (subset.groupby("image_id").size() == captions_per_image).all()
    assert subset["pair_idx"].tolist() == list(range(len(subset)))

    info = {
        "n_pairs": int(len(subset)),
        "n_distinct_images": int(subset["image_id"].nunique()),
        "captions_per_image": captions_per_image,
        "eligible_images": int(len(eligible_ids)),
        "requested_images": int(n_images),
        "short_of_request": int(max(0, n_images - take)),
        "caption_words": {
            "mean": float(subset["n_words"].mean()),
            "min": int(subset["n_words"].min()),
            "max": int(subset["n_words"].max()),
        },
        "seed": seed,
    }
    return subset, info


# ---------------------------------------------------------------------------
# 3. Image preprocessing
# ---------------------------------------------------------------------------
def resize_image(img, size=None):
    """Aspect-preserving resize of the shorter side, then a centre crop.

    A plain `resize((512, 512))` stretches non-square photos, and COCO images are
    mostly 640x480 -- every one would come out horizontally squashed, distorting
    object geometry in a dataset whose whole purpose is object-level semantics.
    `ImageOps.fit` gives exactly size x size with no warping.
    """
    size = cfg.IMAGE_SIZE if size is None else size
    return ImageOps.fit(img, (size, size), method=Image.Resampling.BICUBIC, centering=(0.5, 0.5))


def normalize_pixels(img, value_range="01"):
    """PIL image -> float tensor of shape (C, H, W).

    Two ranges, because different consumers want different ones:
      "01"  -- display, metrics, FID feature extractors
      "pm1" -- the Stable Diffusion VAE encoder
    """
    t = TF.to_tensor(img)                       # [0, 1]
    if value_range == "pm1":
        return t * 2.0 - 1.0                    # same as Normalize(0.5, 0.5)
    if value_range != "01":
        raise ValueError(f"value_range must be '01' or 'pm1', got {value_range!r}")
    return t


def preprocess_image(path, size=None):
    """Load one photo and return a size x size RGB PIL image.

    EXIF orientation is applied first, or a handful of photos arrive rotated 90
    degrees. `convert("RGB")` forces 3 channels, handling greyscale and CMYK.
    """
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    img = img.convert("RGB")
    return resize_image(img, size)


def cache_resized_images(subset, out_dir=None, size=None, quality=95):
    """Process each distinct image once and cache it as JPEG.

    Only the distinct images are processed, not every pair -- the two captions of
    an image share one file. Cached as JPEG rather than as normalised tensors:
    3,000 float32 tensors of shape 3x512x512 would be ~9.4 GB, while the JPEGs are
    a few hundred MB, and normalising at load time is a cheap deterministic step.

    Files already present are skipped, so a re-run costs nothing. Rows whose image
    could not be processed are dropped and `pair_idx` is reassigned, so it stays
    contiguous and keeps matching the embedding arrays.

    Returns `(subset, info)`.
    """
    out_dir = cfg.IMAGES_512_DIR if out_dir is None else Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    unique_images = subset.drop_duplicates("image_id")[["image_id", "file_name", "image_path"]]
    failures, done = [], {}
    t0 = time.time()

    for _, row in tqdm(unique_images.iterrows(), total=len(unique_images), desc="Resizing to 512x512"):
        out_path = out_dir / row["file_name"]
        try:
            if not out_path.exists():
                preprocess_image(row["image_path"], size).save(out_path, "JPEG", quality=quality)
            done[row["image_id"]] = str(out_path)
        except Exception as exc:                  # noqa: BLE001 - record it and keep going
            failures.append((row["file_name"], repr(exc)))

    subset = subset.copy()
    subset["image_512_path"] = subset["image_id"].map(done)
    if failures:
        subset = subset[subset["image_512_path"].notna()].reset_index(drop=True)
        subset["pair_idx"] = np.arange(len(subset))

    info = {
        "n_distinct_images": int(len(unique_images)),
        "n_cached": len(done),
        "failures": failures,
        "seconds": round(time.time() - t0, 1),
        "cache_mb": sum(Path(p).stat().st_size for p in done.values()) / 1024 ** 2,
        "n_pairs_remaining": int(len(subset)),
    }
    return subset, info


def validate_cache(paths, size=None):
    """Confirm every cached file is exactly size x size RGB.

    Returns `(sizes, modes)` as sets, so the caller can assert on them and print
    what it found if the assertion fails.
    """
    size = cfg.IMAGE_SIZE if size is None else size
    sizes, modes = set(), set()
    for p in paths:
        with Image.open(p) as im:
            sizes.add(im.size)
            modes.add(im.mode)
    return sizes, modes

In [ ]:
%%writefile modules/sanity_checks.py
"""Checks that the embeddings are **meaningful**, not merely well-formed.

Matching shapes prove nothing on their own -- a bug that encoded the same caption
6,000 times would pass a shape check. Four checks, cheapest first:

  1. `embedding_health`  expected dimensions, no NaN/Inf, sane vector norms.
  2. `retrieval`         the real test: does each caption retrieve *its own* image
                         out of 500? Chance is 1/500 = 0.2%.
  3. `sibling_gap`       the two captions of one image describe the same scene, so
                         they should land closer together than two random
                         captions. Needs no image tower, so BERT can take part --
                         which is what makes the CLIP-vs-BERT comparison possible.
  4. `top_neighbours`    the qualitative read: what does each encoder think is
                         similar?

These return numbers. Plotting stays in the notebook, where the figures appear.
"""

import numpy as np
import torch
import torch.nn.functional as F

import config as cfg


def expected_shapes(n_rows, save_seq_embeddings=None):
    """The shape every array should have, given the manifest row count."""
    save_seq = cfg.SAVE_SEQ_EMBEDDINGS if save_seq_embeddings is None else save_seq_embeddings
    expected = {
        "clip_text_joint": (n_rows, 512),
        "clip_sd_pooled": (n_rows, 768),
        "bert_cls": (n_rows, 768),
        "bert_mean": (n_rows, 768),
    }
    if save_seq:
        expected["clip_sd_sequence"] = (n_rows, cfg.CLIP_MAX_TOKENS, 768)
    return expected


def embedding_health(arrays, expected):
    """Shape, finiteness and norm statistics per array.

    Returns `(report, all_ok)`.
    """
    report, all_ok = {}, True
    for name, array in arrays.items():
        a32 = array.astype(np.float32)
        shape_ok = array.shape == expected[name]
        finite = bool(np.isfinite(a32).all())
        norms = np.linalg.norm(a32.reshape(len(a32), -1), axis=1)
        all_ok &= shape_ok and finite
        report[name] = {
            "shape": list(array.shape),
            "expected": list(expected[name]),
            "dtype": str(array.dtype),
            "shape_ok": shape_ok,
            "all_finite": finite,
            "norm_mean": float(norms.mean()),
            "norm_std": float(norms.std()),
        }
    return report, bool(all_ok)


def retrieval(text_emb, image_emb):
    """Text -> image retrieval over a square set of caption/image pairs.

    Retrieval needs exactly one correct answer per query, so the caller passes the
    first caption of each image only -- otherwise two queries compete for the same
    target.

    Returns a dict with the metrics plus the raw `sim`, `ranks`, `matched` and
    `mismatched` arrays, so the notebook can plot them without recomputing.
    """
    txt = F.normalize(torch.from_numpy(text_emb.astype(np.float32)), dim=-1)
    img = F.normalize(torch.from_numpy(image_emb.astype(np.float32)), dim=-1)
    sim = (txt @ img.T).numpy()                   # (n, n): rows = captions, cols = images

    n = len(sim)
    ranks = (-sim).argsort(axis=1)
    rank_of_gt = np.array([int(np.where(ranks[i] == i)[0][0]) for i in range(n)])
    r1, r5, r10 = (float((rank_of_gt < k).mean()) for k in (1, 5, 10))
    chance = 1.0 / n

    gt = np.arange(n)
    matched, mismatched = sim[gt, gt], sim[~np.eye(n, dtype=bool)]

    return {
        "n_queries": int(n),
        "recall_at_1": r1, "recall_at_5": r5, "recall_at_10": r10,
        "chance": chance,
        "times_better_than_chance": float(r1 / chance),
        "median_rank_of_correct_image": int(np.median(rank_of_gt)) + 1,
        "cosine_matched_mean": float(matched.mean()),
        "cosine_matched_std": float(matched.std()),
        "cosine_mismatched_mean": float(mismatched.mean()),
        "cosine_mismatched_std": float(mismatched.std()),
        "separation": float(matched.mean() - mismatched.mean()),
        # raw arrays, for plotting
        "sim": sim, "ranks": ranks, "matched": matched, "mismatched": mismatched,
    }


def sibling_gap(encoders, pair_a, pair_b, rng=None):
    """Sibling-caption agreement per encoder.

    A larger gap means the encoder places two descriptions of one scene closer
    together than two unrelated captions -- it is encoding meaning rather than
    surface word overlap. A high sibling cosine with an equally high random cosine
    means the opposite: everything is mapped into a narrow cone, so the rankings
    carry little information. Watch for that on the [CLS] row.

    Returns `{label: {"sibling_cos": .., "random_cos": .., "gap": ..}}`.
    """
    rng = np.random.default_rng(cfg.SEED) if rng is None else rng
    shuffled = rng.permutation(len(pair_b))

    scores = {}
    for label, emb in encoders.items():
        v = F.normalize(torch.from_numpy(emb.astype(np.float32)), dim=-1)
        sib = (v[pair_a] * v[pair_b]).sum(-1).numpy()
        rnd = (v[pair_a] * v[pair_b][shuffled]).sum(-1).numpy()
        scores[label] = {
            "sibling_cos": float(sib.mean()),
            "random_cos": float(rnd.mean()),
            "gap": float(sib.mean() - rnd.mean()),
        }
    return scores


def top_neighbours(emb, query_idx, k=5):
    """The `k` captions nearest to `query_idx` in this encoder's space.

    Returns `(indices, similarities)`. The query itself is excluded.
    """
    v = F.normalize(torch.from_numpy(emb.astype(np.float32)), dim=-1)
    sims = (v[query_idx] @ v.T).numpy()
    sims[query_idx] = -np.inf
    idx = sims.argsort()[::-1][:k]
    return idx, sims[idx]


def clip_scores(prompts, generated_images, real_images, tokenizer, model, processor, device=None):
    """CLIP cosine similarity for a batch of generation runs.

    Returns `(text_vs_generated, text_vs_real, generated_vs_real)`, each (N,).

    Every number is a cosine in CLIP's joint space. Text-image scores sit in a
    narrow band in practice -- roughly 0.25-0.35 for a caption and an image that
    genuinely match, around 0.1 for an unrelated pair. `generated_vs_real` is on a
    different scale entirely and runs much higher, because two images share far
    more structure with each other than a sentence ever shares with an image;
    compare it across rows, never against the first two.
    """
    from .clip_embeddings import encode_images, pooled   # local import: avoids a cycle at import time

    device = cfg.DEVICE if device is None else device
    with torch.no_grad():
        toks = tokenizer(prompts, padding="max_length", max_length=cfg.CLIP_MAX_TOKENS,
                         truncation=True, return_tensors="pt").to(device)
        t_emb = F.normalize(pooled(model.get_text_features(**toks)).float().cpu(), dim=-1)

    g_emb = F.normalize(torch.from_numpy(
        encode_images(generated_images, processor, model, device=device, progress=False)), dim=-1)
    r_emb = F.normalize(torch.from_numpy(
        encode_images(real_images, processor, model, device=device, progress=False)), dim=-1)

    return (
        (t_emb * g_emb).sum(-1).numpy(),
        (t_emb * r_emb).sum(-1).numpy(),
        (g_emb * r_emb).sum(-1).numpy(),
    )

In [ ]:
%%writefile modules/storage.py
"""Everything that touches disk: the manifest, the .npy arrays, metadata, and the
cache logic that lets a re-run skip work it has already done.

The contract these functions enforce: **row *i* of every .npy file is the caption
at `pair_idx == i` in `pairs.parquet`.** One join key, checked before anything is
written and verified by a round-trip read afterwards.

    data/processed/
    |-- pairs.parquet            the curated manifest
    |-- pairs.csv                the same, human-readable
    |-- metadata.json            provenance: models, config, shapes, versions, timings
    |-- images_512/              512x512 RGB JPEG cache
    `-- embeddings/
        |-- clip_text_joint.npy  (N, 512)  float32   shared text/image space
        |-- clip_sd_pooled.npy   (N, 768)  float32   SD v1.5 text encoder, pooled
        |-- bert_cls.npy         (N, 768)  float32   [CLS] token
        `-- bert_mean.npy        (N, 768)  float32   mean-pooled

Figures and summary tables go somewhere separate, under `outputs/`, because they
describe a *run* rather than the dataset itself.

Every path comes from `config.py` -- change `EMB_DIR` there and everything here
follows.
"""

import hashlib
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

import config as cfg
from utils import rel

MANIFEST_COLUMNS = [
    "pair_idx", "image_id", "ann_id", "cap_rank", "file_name",
    "caption_raw", "caption", "n_words", "n_chars",
    "height", "width", "image_path", "image_512_path",
]

# One row per retrieval image rather than one per pair, so it is exempt from the
# row-count check that flags stale arrays.
NON_PAIR_ARRAYS = {"clip_image_align"}


# ---------------------------------------------------------------------------
# Cache validity
# ---------------------------------------------------------------------------
def subset_fingerprint(captions, n_images=None, captions_per_image=None, seed=None):
    """A short hash of the exact caption list plus the settings that shape it.

    Cached arrays are reused only when this matches, so changing `N_IMAGES`, the
    cleaning rules or `SEED` throws the cache out automatically. Without it, a
    re-run could quietly pair new captions with old vectors.
    """
    n_images = cfg.N_IMAGES if n_images is None else n_images
    captions_per_image = cfg.CAPTIONS_PER_IMAGE if captions_per_image is None else captions_per_image
    seed = cfg.SEED if seed is None else seed

    src = "\n".join(captions) + f"|{n_images}|{captions_per_image}|{seed}"
    return hashlib.sha256(src.encode("utf-8")).hexdigest()[:16]


def cache_is_usable(fingerprint, previous_metadata=None, reuse=None, force_recompute=None):
    """Whether cached embeddings may be reused for this run."""
    reuse = cfg.REUSE_EXISTING if reuse is None else reuse
    force_recompute = cfg.FORCE_RECOMPUTE if force_recompute is None else force_recompute
    previous = (previous_metadata or {}).get("subset_fingerprint")
    return bool(reuse and not force_recompute and previous == fingerprint)


# ---------------------------------------------------------------------------
# Embedding arrays
# ---------------------------------------------------------------------------
def embedding_path(name, emb_dir=None):
    return (cfg.EMB_DIR if emb_dir is None else Path(emb_dir)) / f"{name}.npy"


def save_embedding(name, array, emb_dir=None):
    path = embedding_path(name, emb_dir)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, array)
    return path


def load_embedding(name, emb_dir=None, mmap=False):
    return np.load(embedding_path(name, emb_dir), mmap_mode="r" if mmap else None)


def load_or_encode(names, encode_fn, n_rows, cache_usable=False, save=True,
                   reuse_log=None, emb_dir=None):
    """Return the named arrays, loading them from disk when a valid cache exists.

    `encode_fn` is only called on a miss, and its results are saved immediately so
    that a failure further down never throws away encoding work.

    Returns `(*arrays, elapsed_seconds)`; elapsed is 0.0 on a cache hit.
    """
    reuse_log = {} if reuse_log is None else reuse_log
    paths = [embedding_path(n, emb_dir) for n in names]

    if cache_usable and all(p.is_file() for p in paths):
        arrays = [np.load(p) for p in paths]
        if all(len(a) == n_rows for a in arrays):
            print(f"Reusing cached: {', '.join(p.name for p in paths)}")
            reuse_log.update({n: "reused" for n in names})
            return (*arrays, 0.0)
        print("  cached arrays have the wrong row count -- recomputing")

    t0 = time.time()
    result = encode_fn()
    arrays = list(result) if isinstance(result, tuple) else [result]
    elapsed = time.time() - t0

    for name, array in zip(names, arrays):
        if save:
            save_embedding(name, array, emb_dir)
            reuse_log[name] = "computed"
        else:
            reuse_log[name] = "computed (not saved)"
    return (*arrays, elapsed)


def save_arrays(arrays, manifest, emb_dir=None):
    """Write every array, each row-checked against the manifest first.

    Returns a list of `(name, shape, dtype, megabytes)` for the notebook to print.
    """
    rows = []
    for name, array in arrays.items():
        assert len(array) == len(manifest), (
            f"{name} has {len(array)} rows but the manifest has {len(manifest)}"
        )
        save_embedding(name, array, emb_dir)
        rows.append((name, array.shape, str(array.dtype), array.nbytes / 1024 ** 2))
    return rows


def find_stale(arrays, n_rows, emb_dir=None):
    """Arrays on disk left over from an earlier run with a different subset size.

    Reported, never deleted -- throwing away someone else's work on a path guess
    is not this function's call to make.
    """
    emb_dir = cfg.EMB_DIR if emb_dir is None else Path(emb_dir)
    stale = []
    for p in sorted(emb_dir.glob("*.npy")):
        if p.stem in arrays or p.stem in NON_PAIR_ARRAYS:
            continue
        shape = np.load(p, mmap_mode="r").shape
        if shape[0] != n_rows:
            stale.append((rel(p), shape, p.stat().st_size / 1024 ** 2))
    return stale


def existing_embeddings(emb_dir=None):
    """What is already on disk, for the "what will this run cost" report."""
    emb_dir = cfg.EMB_DIR if emb_dir is None else Path(emb_dir)
    if not emb_dir.is_dir():
        return []
    return [
        (p.name, np.load(p, mmap_mode="r").shape, p.stat().st_size / 1024 ** 2)
        for p in sorted(emb_dir.glob("*.npy"))
    ]


# ---------------------------------------------------------------------------
# Manifest
# ---------------------------------------------------------------------------
def write_manifest(subset, parquet_path=None, csv_path=None):
    """Write the manifest in the exact row order of the embedding arrays.

    Image paths are stored relative to the project root, so the artefacts stay
    portable across machines.
    """
    parquet_path = cfg.MANIFEST_PARQUET if parquet_path is None else Path(parquet_path)
    csv_path = cfg.MANIFEST_CSV if csv_path is None else Path(csv_path)
    parquet_path.parent.mkdir(parents=True, exist_ok=True)

    manifest = subset[MANIFEST_COLUMNS].copy()
    for col in ("image_path", "image_512_path"):
        manifest[col] = manifest[col].map(rel)

    manifest.to_parquet(parquet_path, index=False)
    manifest.to_csv(csv_path, index=False, encoding="utf-8")
    return manifest


def read_manifest(parquet_path=None):
    return pd.read_parquet(cfg.MANIFEST_PARQUET if parquet_path is None else parquet_path)


# ---------------------------------------------------------------------------
# Metadata
# ---------------------------------------------------------------------------
def load_metadata(path=None):
    """The previous run's metadata, or None if there is none."""
    path = cfg.METADATA_FILE if path is None else Path(path)
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def save_metadata(metadata, path=None):
    path = cfg.METADATA_FILE if path is None else Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    return path


def build_metadata(manifest, arrays, dataset_counts, cleaning_steps, timings,
                   fingerprint, reuse_log, not_persisted=None):
    """Assemble the provenance record.

    The config-derived sections are read straight from `config.py`; the caller
    supplies only the numbers that come out of the run itself.
    """
    import diffusers
    import torch
    import transformers

    return {
        "dataset": {
            "name": "COCO 2017 Captions",
            "split": "val2017",
            "annotations_file": rel(cfg.ANN_FILE),
            **dataset_counts,
            "n_curated_pairs": int(len(manifest)),
            "n_distinct_images": int(manifest["image_id"].nunique()),
            "captions_per_image": cfg.CAPTIONS_PER_IMAGE,
        },
        "image_preprocessing": {
            "size": [cfg.IMAGE_SIZE, cfg.IMAGE_SIZE],
            "mode": "RGB",
            "resize": "aspect-preserving shorter-side resize + centre crop (PIL ImageOps.fit, bicubic)",
            "exif_transpose": True,
            "normalisations": {"display_and_metrics": "[0, 1]", "sd_vae": "[-1, 1] via Normalize(0.5, 0.5)"},
            "cache_dir": rel(cfg.IMAGES_512_DIR),
        },
        "caption_cleaning": {
            "steps": list(cleaning_steps),
            "filters": {
                "min_words": cfg.MIN_WORDS, "max_words": cfg.MAX_WORDS,
                "min_chars": cfg.MIN_CHARS, "max_chars": cfg.MAX_CHARS,
                "drop_empty": True, "drop_duplicates": True, "drop_missing_image": True,
            },
            "stopwords": "not removed -- see the note in section 4",
        },
        "embeddings": {n: {"shape": list(a.shape), "dtype": str(a.dtype)} for n, a in arrays.items()},
        "embeddings_not_persisted": not_persisted or {},
        "models": {
            "clip_joint": cfg.CLIP_JOINT_MODEL,
            "clip_sd_text_encoder": cfg.CLIP_SD_MODEL,
            "bert": cfg.BERT_MODEL,
            "clip_max_tokens": cfg.CLIP_MAX_TOKENS,
            "bert_max_tokens": cfg.BERT_MAX_TOKENS,
            "encoder_dtype": str(cfg.ENCODER_DTYPE),
        },
        "encode_seconds": {k: round(v, 1) for k, v in timings.items()},
        "versions": {
            "python": sys.version.split()[0],
            "torch": torch.__version__,
            "transformers": transformers.__version__,
            "diffusers": diffusers.__version__,
        },
        "seed": cfg.SEED,
        "subset_fingerprint": fingerprint,   # guards cache reuse on the next run
        "cache_status": dict(reuse_log),
    }


# ---------------------------------------------------------------------------
# Round trip
# ---------------------------------------------------------------------------
def verify_round_trip(manifest, arrays, check=("clip_text_joint", "bert_mean")):
    """Reload from disk and confirm it matches what is in memory.

    Without this, a silent write failure would only surface in a later notebook.
    Returns a small summary dict; raises AssertionError if anything disagrees.
    """
    reloaded = read_manifest()
    assert len(reloaded) == len(manifest), "manifest row count changed on the round trip"
    assert reloaded["pair_idx"].tolist() == list(range(len(reloaded))), "pair_idx is not contiguous"

    for name in check:
        on_disk = load_embedding(name)
        assert np.array_equal(on_disk, arrays[name]), f"{name} does not match what was written"

    probe = reloaded.iloc[min(7, len(reloaded) - 1)]
    return {
        "manifest_rows": int(len(reloaded)),
        "arrays_checked": list(check),
        "probe_caption": probe["caption"],
        "probe_image": probe["image_512_path"],
    }

In [ ]:
import sys

sys.path.insert(0, ".")
for _name in [m for m in sys.modules if m in ("config", "utils") or m.startswith("modules")]:
    del sys.modules[_name]          # so a re-run picks up the freshly written files

import config as cfg
import utils
from modules import bert_embeddings, clip_embeddings, data_preprocessing, sanity_checks, storage

print("modules ready:", 5)

---
## 1. Environment Setup

The stack is Python + PyTorch, Hugging Face **`diffusers`** for the diffusion pipeline and Hugging Face
**`transformers`** for the text encoders. Dependencies are declared in `pyproject.toml` and installed
with `uv sync`, so this section only **verifies** what is installed rather than installing anything —
the versions printed below are reproducible rather than whatever happens to be installed globally.

The first cell also puts `src/` on the import path. It searches upward from the working directory, so
it works from `src/notebooks/`, from the repo root, and on Kaggle wherever the repo is attached —
without `pip install -e .`, which needs the internet toggle that a Kaggle session may have switched off.


In [ ]:
import json, time, warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import transformers
import diffusers

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()   # silence the per-model weight-loading reports

print("TOOLCHAIN")
print("-" * 46)
for name, version in [
    ("python", sys.version.split()[0]),
    ("torch", torch.__version__),
    ("transformers", transformers.__version__),
    ("diffusers", diffusers.__version__),
    ("numpy", np.__version__),
    ("pandas", pd.__version__),
    ("Pillow", Image.__version__),
]:
    print(f"  {name:<16}{version}")


---
## 2. Configuration, Paths & Reproducibility

Every tunable value lives in **`src/config.py`** — paths, subset size, caption filters, model names,
generation settings, toggles. Nothing in this notebook hard-codes a number that matters. To re-run the
pipeline at a different scale, edit that file, restart the kernel, and run from the top; no code below
needs touching.

`config.py` also reads three environment variables, which is how the same notebook runs on Kaggle
without edits: `T2I_PROJECT_ROOT`, `T2I_DATA_DIR`, `T2I_OUTPUT_DIR`.

A single `SEED` makes the subset, the sampled examples and the generated images identical on every run.

**Subset: 3,000 images × 2 captions = 6,000 pairs.** The alternatives were 2,500 × 2 and 5,000 × 1. We
take 3,000 × 2 to keep image variety and caption variety in balance: more distinct scenes than 2,500 × 2,
and two human phrasings of each scene rather than one.

**Generator: Stable Diffusion v1.5**, chosen because compute is limited — a single Kaggle T4 with a 9-hour
session cap. It is the smallest of the usable text-to-image checkpoints, runs at 512×512 inside a modest
VRAM budget, and its CLIP text encoder is the one section 7 extracts conditioning embeddings from.


In [ ]:
import config as cfg
import utils
from modules import data_preprocessing as prep
from modules import clip_embeddings, bert_embeddings, storage, sanity_checks

cfg.ensure_dirs()
utils.set_seed()

# Verify the inputs are where config says they are.
assert cfg.ANN_FILE.is_file(), f"Missing annotations file: {cfg.ANN_FILE}"
assert cfg.IMAGES_DIR.is_dir(), f"Missing images directory: {cfg.IMAGES_DIR}"
n_jpg = sum(1 for _ in cfg.IMAGES_DIR.glob("*.jpg"))

print(f"Project root  : {cfg.PROJECT_ROOT}")
print(f"Annotations   : {utils.rel(cfg.ANN_FILE)}  ({cfg.ANN_FILE.stat().st_size / 1024**2:.1f} MB)")
print(f"Images        : {utils.rel(cfg.IMAGES_DIR)}  ({n_jpg:,} .jpg files)")
print(f"Target subset : {cfg.N_IMAGES:,} images x {cfg.CAPTIONS_PER_IMAGE} captions = "
      f"{cfg.N_IMAGES * cfg.CAPTIONS_PER_IMAGE:,} pairs")
print(f"Seed          : {cfg.SEED}")
print()
print(f"Device        : {'GPU' if cfg.DEVICE.type == 'cuda' else 'CPU'} ({cfg.DEVICE})")
print(f"Encoder dtype : {cfg.ENCODER_DTYPE}   SD dtype: {cfg.SD_DTYPE}")


### Saving what we measure

Figures and summary tables are written to disk as well as shown inline, so the numbers behind this
notebook can be read later without re-running it — `outputs/plots/` for figures, `outputs/tables/` for
summary tables as JSON. `utils.save_fig` and `utils.save_table` do it, and every plot and table below
goes through them.

### What is already on disk

This notebook is **resumable**: model weights, resized images and embedding arrays are all reused when
they are already present and still valid. The cell below reports what a re-run will skip, so the cost of
running it is visible before anything starts.


In [ ]:
from huggingface_hub import scan_cache_dir

try:
    hf_cache = {r.repo_id: r.size_on_disk for r in scan_cache_dir().repos}
except Exception as exc:                      # noqa: BLE001 - the cache may not exist yet
    print(f"(could not scan the Hugging Face cache: {type(exc).__name__})")
    hf_cache = {}

print("MODEL WEIGHTS")
print("-" * 72)
for repo in (cfg.CLIP_JOINT_MODEL, cfg.CLIP_SD_MODEL, cfg.BERT_MODEL):
    size = hf_cache.get(repo)
    print(f"  {repo:<44}{f'cached ({size / 1024**2:,.0f} MB)' if size else 'WILL DOWNLOAD'}")

sd_cached = next((r for r in cfg.SD_REPO_CANDIDATES if r in hf_cache), None)
if sd_cached:
    print(f"  {sd_cached:<44}cached ({hf_cache[sd_cached] / 1024**2:,.0f} MB)")
else:
    print(f"  {'stable-diffusion-v1-5':<44}WILL DOWNLOAD (~4 GB in section 11)")

print()
print("PROCESSED ARTEFACTS")
print("-" * 72)
print(f"  images_512/{'':<33}{sum(1 for _ in cfg.IMAGES_512_DIR.glob('*.jpg')):,} files")
existing = storage.existing_embeddings()
for name, shape, mb in existing:
    print(f"  embeddings/{name:<33}{str(shape):<18}{mb:>7.1f} MB")
if not existing:
    print(f"  embeddings/{'':<33}(none yet)")

prev_meta = storage.load_metadata()
print(f"  metadata.json{'':<31}{'present' if prev_meta else 'none yet'}")
print()
print(f"  REUSE_EXISTING = {cfg.REUSE_EXISTING}   FORCE_RECOMPUTE = {cfg.FORCE_RECOMPUTE}   "
      f"RUN_SD_GENERATION = {cfg.RUN_SD_GENERATION}")


---
## 3. Dataset Loading

**COCO 2017 Captions**, the `val2017` split — a paired image–caption dataset, which is what a
text-to-image model needs: a sentence and the picture it describes.

We use the validation split rather than `train2017` because it is already enough. 5,000 images with about
five captions each is roughly **25,000 candidate pairs**, and we need 6,000 of them. `train2017` is an
18 GB download for data this project would never touch, and nothing here is trained, so there is no
train/test split to respect.

The annotation file follows the standard COCO schema — `images -> [{id, file_name, height, width, ...}]`
and `annotations -> [{id, image_id, caption}]` — which we join on `image_id`. Loading is a handful of
pandas calls specific to this one file, so it stays here rather than becoming a module.


In [ ]:
t0 = time.time()
with open(cfg.ANN_FILE, "r", encoding="utf-8") as f:
    coco = json.load(f)

print(f"Loaded {cfg.ANN_FILE.name} in {time.time() - t0:.1f}s")
print(f"  top-level keys : {list(coco.keys())}")
print()
print(f"Example image record   : {coco['images'][0]}")
print()
print(f"Example caption record : {coco['annotations'][0]}")


In [ ]:
# Join captions to their image metadata on image_id.
images_df = pd.DataFrame(coco["images"])[["id", "file_name", "height", "width"]]
images_df = images_df.rename(columns={"id": "image_id"})

anns_df = pd.DataFrame(coco["annotations"])[["id", "image_id", "caption"]]
anns_df = anns_df.rename(columns={"id": "ann_id", "caption": "caption_raw"})

pairs_all = anns_df.merge(images_df, on="image_id", how="inner")
pairs_all["image_path"] = pairs_all["file_name"].map(lambda fn: str(cfg.IMAGES_DIR / fn))

# Sort by a stable key so the row order -- and therefore the subset -- is identical on every run.
pairs_all = pairs_all.sort_values(["image_id", "ann_id"]).reset_index(drop=True)

n_images, n_captions = pairs_all["image_id"].nunique(), len(pairs_all)
print("Source pool")
print(f"  distinct images           : {n_images:,}")
print(f"  caption annotations       : {n_captions:,}")
print(f"  captions per image (mean) : {n_captions / n_images:.2f}")
print()
print(f"  candidate (image, caption) pairs : {n_captions:,}")
print("  one pair per caption, so the pair count follows the caption count, not the image count.")
pairs_all.head(5)


In [ ]:
# Shape of the corpus before any cleaning -- the baseline the next section is measured against.
raw_words = pairs_all["caption_raw"].str.split().str.len()
raw_chars = pairs_all["caption_raw"].str.len()

raw_stats = {
    "n_candidate_pairs": int(len(pairs_all)),
    "n_distinct_images": int(pairs_all["image_id"].nunique()),
    "captions_per_image_mean": float(len(pairs_all) / pairs_all["image_id"].nunique()),
    "words": {"min": int(raw_words.min()), "median": float(raw_words.median()), "max": int(raw_words.max())},
    "chars": {"min": int(raw_chars.min()), "median": float(raw_chars.median()), "max": int(raw_chars.max())},
    "exact_duplicate_captions": int(pairs_all["caption_raw"].duplicated().sum()),
}

print("Raw caption statistics")
print(f"  words : min {raw_words.min():3d} | median {raw_words.median():5.1f} | max {raw_words.max():3d}")
print(f"  chars : min {raw_chars.min():3d} | median {raw_chars.median():5.1f} | max {raw_chars.max():3d}")
print(f"  exact duplicate captions : {pairs_all['caption_raw'].duplicated().sum():,}")
print(f"  saved -> {utils.save_table('raw_caption_stats', raw_stats)}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].hist(raw_words, bins=40, color="#4C72B0")
axes[0].set_title("Raw caption length (words)")
axes[0].set_xlabel("words")
axes[1].hist(pairs_all.groupby("image_id").size(), bins=range(1, 10), color="#DD8452", align="left")
axes[1].set_title("Captions per image")
axes[1].set_xlabel("captions")
plt.tight_layout()
print(f"saved -> {utils.save_fig(fig, 'raw_caption_stats')}")
plt.show()


---
## 4. Caption Cleaning

→ `src/modules/data_preprocessing.py` — `clean_caption`, `clean_captions`, `filter_captions`

To get the captions ready for the text encoders we lowercase and strip them, remove special characters
and HTML artifacts, drop the ones that are empty, duplicated or too short or long, and normalise
whitespace and any character that would break tokenisation.

`clean_caption` does this with **regular expressions and libraries rather than hand-written character
lists**, which matters for coverage: a blacklist only removes the characters someone thought of, whereas
a **whitelist** regex (`[^a-z0-9\s'.,!?-]` → space) removes every character we did *not* think of,
including emoji, currency symbols and any stray Unicode in the corpus. Two libraries do the work that
would otherwise be hand-coded: **`ftfy`** repairs mojibake, and Python's **`html`** module unescapes
entities.

Order matters, because each step assumes the previous one has run:

| # | Step | Library / pattern | Example |
|---|------|-------------------|---------|
| 1 | mojibake repair | `ftfy.fix_text` | `cafÃ©` → `café` |
| 2 | HTML unescape + tag strip | `html.unescape`, regex | `&amp;` → `&`, `<br/>` → ` ` |
| 3 | URL removal | regex | `http://…` → ` ` |
| 4 | typographic folding | 3 regexes | `’ “ ” —` → `' " " -` |
| 5 | Unicode NFKD + ASCII fold | `unicodedata` | `café` → `cafe` |
| 6 | lowercase + strip | `str` | `A MAN ` → `a man` |
| 7 | **whitelist** regex | `[^a-z0-9\s'.,!?-]` | `sym©bols` → `symbols` |
| 8 | punctuation collapse | 2 regexes | `!!!` → `!`, `cat .` → `cat.` |
| 9 | whitespace normalisation | `\s+` → `" "` | `a\tb\nc\xa0d` → `a b c d` |

**We considered removing stop words with NLTK, and decided against it.** CLIP and BERT were both
pretrained on ordinary English containing words like "a", "on" and "with", so removing them would not
help the embeddings and would cost us meaning we need — *"a cat **on** a laptop"* and *"a cat **under** a
laptop"* are two different pictures.


In [ ]:
# Every rule above, demonstrated on deliberately messy input.
demo = [
    "A  MAN   riding a HORSE on the beach.",
    "Two cats &amp; a dog <br/> sleeping.",
    "The caf\u00e9\u2019s sign reads \u201cOPEN\u201d \u2014 see http://example.com/menu",
    "What a view!!!   Really ???",
    "A\tplate\nof\xa0food   .",
    "Emoji \U0001F355 and sym\u00a9bols \u00bd removed",
    "   ",
]
print(f"{'RAW':<62} -> CLEANED")
print("-" * 104)
for s in demo:
    print(f"{s!r:<62} -> {prep.clean_caption(s)!r}")


In [ ]:
# Apply the cleaner to all ~25k candidate captions.
pairs_all = prep.clean_captions(pairs_all)

# How much of the cleaner is load-bearing on THIS corpus, as opposed to defensive?
rule_hits = prep.cleaning_rule_hits(pairs_all)

print()
print(f"Cleaned {rule_hits['n_captions']:,} captions")
print(f"  altered by cleaning       : {rule_hits['altered_by_cleaning']:,}")
print()
print("Which rules actually fire on this corpus")
print(f"  captions with non-ASCII characters : {rule_hits['with_non_ascii']:,}  (steps 1, 4, 5 are defensive)")
print(f"  captions with HTML entities/tags   : {rule_hits['with_html']:,}")
print(f"  captions with whitespace to fix    : {rule_hits['with_whitespace_to_fix']:,}")
print(f"  captions needing lowercasing       : {rule_hits['needing_lowercase']:,}")
print(f"  saved -> {utils.save_table('cleaning_rule_hits', rule_hits)}")


In [ ]:
# Filter, reporting how many rows each rule is responsible for removing. Doing this as a table rather
# than one chained expression makes it obvious if a single rule is dropping too much.
pairs_clean, stages = prep.filter_captions(pairs_all)

prev = None
print(f"{'STAGE':<36}{'REMAINING':>12}{'REMOVED':>10}")
print("-" * 58)
for name, n in stages:
    print(f"{name:<36}{n:>12,}{'' if prev is None else f'{prev - n:,}':>10}")
    prev = n

print()
print(f"Surviving captions : {len(pairs_clean):,} across {pairs_clean['image_id'].nunique():,} images")
filter_report = {
    "stages": [{"stage": name, "remaining": int(n)} for name, n in stages],
    "surviving_captions": int(len(pairs_clean)),
    "surviving_images": int(pairs_clean["image_id"].nunique()),
}
print(f"saved -> {utils.save_table('caption_filter_stages', filter_report)}")


In [ ]:
# Cleaning shown against the image each caption describes, so the text can be checked for whether it
# still says what the picture shows.
changed = pairs_clean[pairs_clean["caption"] != pairs_clean["caption_raw"]]
examples = changed.sample(4, random_state=cfg.SEED).reset_index(drop=True)

fig, axes = plt.subplots(1, 4, figsize=(15, 4.6))
for ax, (_, row) in zip(axes, examples.iterrows()):
    ax.imshow(ImageOps.exif_transpose(Image.open(row["image_path"])).convert("RGB"))
    ax.axis("off")
    wrap = lambda s: "\n".join([s[i:i + 34] for i in range(0, min(len(s), 102), 34)])
    ax.set_title(f"RAW\n{wrap(row['caption_raw'])}\n\nCLEANED\n{wrap(row['caption'])}", fontsize=7, loc="left")

fig.suptitle("Caption cleaning, against the image each caption describes", y=1.02)
plt.tight_layout()
print(f"saved -> {utils.save_fig(fig, 'caption_cleaning_examples')}")
plt.show()

for _, row in examples.iterrows():
    print(f"  {row['file_name']}")
    print(f"    raw     : {row['caption_raw']!r}")
    print(f"    cleaned : {row['caption']!r}")


---
## 5. Subset Construction

→ `src/modules/data_preprocessing.py` — `build_subset`

From the cleaned pool we keep **3,000 distinct images with 2 captions each = 6,000 image–caption pairs**,
for the reasons set out in section 2.

Selection is deterministic: captions are ranked within each image by annotation id, images with at least
two surviving captions are eligible, and `N_IMAGES` of them are drawn with a seeded generator. Re-running
the notebook therefore produces exactly the same 6,000 rows.

Each row gets a `pair_idx` — `0 … 5,999` — which becomes the join key between the manifest and every
embedding array written in section 9. Row *i* of any `.npy` file is the caption on row *i* of the
manifest; nothing else needs to be tracked.


In [ ]:
subset, subset_stats = prep.build_subset(pairs_clean)

if subset_stats["short_of_request"]:
    print(f"WARNING: only {subset_stats['n_distinct_images']:,} images are eligible, "
          f"fewer than the requested {cfg.N_IMAGES:,}")

print(f"Images with >= {cfg.CAPTIONS_PER_IMAGE} surviving captions : {subset_stats['eligible_images']:,}")
print(f"Curated subset : {subset_stats['n_pairs']:,} image-caption pairs")
print(f"  distinct images    : {subset_stats['n_distinct_images']:,}")
print(f"  captions per image : {subset_stats['captions_per_image']}")
print(f"  caption length     : mean {subset_stats['caption_words']['mean']:.1f} words | "
      f"min {subset_stats['caption_words']['min']} | max {subset_stats['caption_words']['max']}")
print(f"  saved -> {utils.save_table('subset_summary', subset_stats)}")
subset[["pair_idx", "image_id", "cap_rank", "file_name", "caption"]].head(6)


---
## 6. Image Preprocessing

→ `src/modules/data_preprocessing.py` — `preprocess_image`, with `resize_image` and `normalize_pixels`
underneath it, and `cache_resized_images` to run it over the subset

We resize every image to **512×512**, the resolution Stable Diffusion v1.x works at natively, normalise
the pixel values to the ranges the rest of the pipeline expects, and force a **consistent RGB 3-channel
format** so nothing downstream has to special-case a greyscale photo.

**Resize strategy.** A plain `resize((512, 512))` stretches non-square photos, and COCO images are mostly
640×480 — every one of them would come out horizontally squashed, distorting object geometry in a dataset
whose whole purpose is object-level semantics. Instead `resize_image` does an aspect-preserving resize of
the shorter side followed by a centre crop (`PIL.ImageOps.fit`), giving exactly 512×512 with no warping.
EXIF orientation is applied first, or a handful of photos arrive rotated 90°.

**Two normalisations**, because different consumers want different ranges:

| Range | `normalize_pixels(img, ...)` | Consumed by |
|-------|------------------------------|-------------|
| `[0, 1]` | `value_range="01"` | display, metrics, FID feature extractors |
| `[-1, 1]` | `value_range="pm1"` | the Stable Diffusion VAE encoder |

**What is cached, and why not tensors.** Only the **3,000 distinct images** are processed, not 6,000 — the
two captions of an image share one file. We cache them as JPEGs rather than as normalised tensors: 3,000
float32 tensors of shape `3×512×512` would be ~9.4 GB, while the JPEGs are a few hundred MB, and applying
the normalisation at load time is a cheap deterministic transform.


In [ ]:
# Verify both transforms on a single image before processing 3,000 of them.
probe_path = subset.iloc[0]["image_path"]
probe_raw = Image.open(probe_path)
probe_512 = prep.preprocess_image(probe_path)
t01 = prep.normalize_pixels(probe_512, "01")
tpm = prep.normalize_pixels(probe_512, "pm1")

print(f"Original  : {probe_raw.size} px, mode={probe_raw.mode}")
print(f"Processed : {probe_512.size} px, mode={probe_512.mode}")
print()
print(f"[0,1] tensor  : shape {tuple(t01.shape)}, dtype {t01.dtype}, range [{t01.min():.3f}, {t01.max():.3f}]")
print(f"[-1,1] tensor : shape {tuple(tpm.shape)}, dtype {tpm.dtype}, range [{tpm.min():.3f}, {tpm.max():.3f}]")

assert t01.shape == (3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)
assert 0.0 <= t01.min() and t01.max() <= 1.0
assert -1.0 <= tpm.min() and tpm.max() <= 1.0
print()
print("Both normalisations verified on a single image.")


In [ ]:
# Before / after, to confirm the centre crop is keeping the subject rather than cutting it off.
show = subset.drop_duplicates("image_id").sample(4, random_state=cfg.SEED)
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for col, (_, row) in enumerate(show.iterrows()):
    orig = ImageOps.exif_transpose(Image.open(row["image_path"])).convert("RGB")
    axes[0, col].imshow(orig)
    axes[0, col].set_title(f"original {orig.size[0]}x{orig.size[1]}", fontsize=9)
    axes[1, col].imshow(prep.preprocess_image(row["image_path"]))
    axes[1, col].set_title(f"processed {cfg.IMAGE_SIZE}x{cfg.IMAGE_SIZE}", fontsize=9)
    for r in (0, 1):
        axes[r, col].axis("off")

fig.suptitle("Image preprocessing: aspect-preserving resize + centre crop", y=0.98)
plt.tight_layout()
print(f"saved -> {utils.save_fig(fig, 'image_preprocessing_examples')}")
plt.show()

for _, row in show.iterrows():
    print(f"  {row['file_name']} : {row['caption']}")


In [ ]:
# Process each distinct image once and cache the result. Files already present are skipped, so a
# re-run costs nothing.
subset, cache_info = prep.cache_resized_images(subset)

print()
print(f"Processed {cache_info['n_cached']:,}/{cache_info['n_distinct_images']:,} distinct images "
      f"in {cache_info['seconds']:.1f}s")
if cache_info["failures"]:
    print(f"Failures ({len(cache_info['failures'])}):")
    for name, err in cache_info["failures"][:10]:
        print(f"  {name}: {err}")

print(f"Cache size: {cache_info['cache_mb']:,.0f} MB in {utils.rel(cfg.IMAGES_512_DIR)}")
print(f"Pairs remaining: {cache_info['n_pairs_remaining']:,}")


In [ ]:
# Validate the cache: every cached file must be exactly 512x512 RGB, with pixels in the stated ranges.
check = subset.drop_duplicates("image_id").sample(
    min(200, subset["image_id"].nunique()), random_state=cfg.SEED)
sizes, modes = prep.validate_cache(check["image_512_path"])

print(f"Validated {len(check)} cached images")
print(f"  distinct sizes : {sizes}")
print(f"  distinct modes : {modes}")
assert sizes == {(cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)}, f"Unexpected sizes: {sizes}"
assert modes == {"RGB"}, f"Unexpected modes: {modes}"

batch01 = torch.stack([prep.normalize_pixels(Image.open(p), "01")
                       for p in check["image_512_path"][:64]])
batchpm = batch01 * 2.0 - 1.0
print()
print(f"Sample batch shape : {tuple(batch01.shape)}  (N, C, H, W)")
print(f"  [0,1]  range {batch01.min():.3f} .. {batch01.max():.3f} | mean {batch01.mean():.3f} | std {batch01.std():.3f}")
print(f"  [-1,1] range {batchpm.min():.3f} .. {batchpm.max():.3f} | mean {batchpm.mean():.3f} | std {batchpm.std():.3f}")
print()
print("All sampled images are 512x512 RGB with pixel values in the expected ranges.")


---
## 7. Text Embeddings — CLIP

→ `src/modules/clip_embeddings.py`, with caching from `src/modules/storage.py`

CLIP was trained on hundreds of millions of image–caption pairs with a **contrastive** objective: pull a
matching image and caption together in a shared space, push mismatched ones apart. Its text vectors
therefore live in a space organised by *visual* meaning, which is precisely what conditions a
text-to-image model — and it is why Stable Diffusion v1.5 uses a CLIP text encoder rather than a
general-purpose language model.

**Why we load two CLIP checkpoints.** They do two different jobs.

The first is **conditioning**: SD v1.5 was built on top of one specific text encoder,
`clip-vit-large-patch14`, and only understands vectors from that encoder. No substitutes.

The second is **checking our own data**: confirming a caption really does match the image it is paired
with needs a model that turns *both* text and images into vectors in one shared space.
`clip-vit-base-patch32` carries both towers, so a caption can be compared against a photo directly. It is
also the cheaper model, which matters since section 10 puts 500 images through it.

| Checkpoint | Output | Why we need it |
|---|---|---|
| `clip-vit-base-patch32` | pooled **512-d** joint-space vector | has a **vision tower too**, which is what makes the text↔image retrieval check in section 10 possible |
| `clip-vit-large-patch14` | **77 × 768** per-token states | the exact encoder SD v1.5 is conditioned on; the tensors the UNet cross-attends to |

The conditioning tensor is **77 × 768** — one vector per token, not one for the whole sentence — because
the generator reads the prompt token by token. That is how "a **red** bus" colours the bus instead of
tinting the whole picture red.

**Caching.** `storage.subset_fingerprint` hashes the embeddings against the exact caption list that
produced them, so changing `N_IMAGES`, the cleaning rules or `SEED` throws the cache out automatically.
Without that, a re-run could quietly pair new captions with old vectors.


In [ ]:
captions = subset["caption"].tolist()

FINGERPRINT = storage.subset_fingerprint(captions)
CACHE_USABLE = storage.cache_is_usable(FINGERPRINT, prev_meta)
reuse_log = {}

print(f"Captions to encode : {len(captions):,}")
print(f"Subset fingerprint : {FINGERPRINT}")
print(f"Previous run       : {(prev_meta or {}).get('subset_fingerprint') or '(none)'}")
print(f"Cached embeddings reusable : {CACHE_USABLE}")
if prev_meta and prev_meta.get("subset_fingerprint") not in (None, FINGERPRINT):
    print("  -> the subset changed since the last run, so embeddings will be recomputed")


In [ ]:
# The joint-space model always loads: sections 10 and 11 need its vision tower regardless of caching.
utils.set_seed()
clip_tok, clip_model, clip_proc = clip_embeddings.load_joint()

print(f"Loaded {cfg.CLIP_JOINT_MODEL}")
for k, v in clip_embeddings.describe(clip_model).items():
    print(f"  {k:<22}{v:,.0f}" if isinstance(v, float) else f"  {k:<22}{v}")


In [ ]:
# CLIP's positional table stops at 77 tokens, so anything longer is silently truncated -- the model
# simply never sees the end of the caption. Measure where our captions sit against that ceiling, and
# check whether our word-count filter is already enough to keep us under it.
tok_len = clip_embeddings.token_lengths(captions, clip_tok)
n_truncated = int((tok_len > cfg.CLIP_MAX_TOKENS).sum())
p50, p95, p99 = (float(np.percentile(tok_len, q)) for q in (50, 95, 99))
headroom = cfg.CLIP_MAX_TOKENS - int(tok_len.max())

token_stats = {
    "clip_max_tokens": cfg.CLIP_MAX_TOKENS,
    "min": int(tok_len.min()), "mean": float(tok_len.mean()),
    "p50": p50, "p95": p95, "p99": p99, "max": int(tok_len.max()),
    "n_truncated": n_truncated,
    "pct_truncated": float(100 * n_truncated / len(tok_len)),
    "headroom_tokens": headroom,
    "max_words_filter": cfg.MAX_WORDS,
}

print(f"CLIP token budget  : {cfg.CLIP_MAX_TOKENS} tokens")
print(f"Our captions       : min {tok_len.min()} | median {p50:.0f} | p95 {p95:.0f} | "
      f"p99 {p99:.0f} | max {tok_len.max()}")
print(f"Over the limit     : {n_truncated} captions ({100 * n_truncated / len(tok_len):.2f}%)")
print(f"Headroom left      : {headroom} tokens below the ceiling")
print()
print("What this means for cleaning:")
if n_truncated == 0:
    print(f"  No caption is truncated, so MAX_WORDS = {cfg.MAX_WORDS} is already a safe proxy for the")
    print(f"  {cfg.CLIP_MAX_TOKENS}-token limit -- the longest caption uses "
          f"{tok_len.max()} tokens for {subset['n_words'].max()} words.")
    print("  No token-based filter is needed on this corpus.")
else:
    print(f"  {n_truncated} captions lose their tail. Lowering MAX_WORDS, or filtering on token count")
    print(f"  directly, would keep every caption inside the {cfg.CLIP_MAX_TOKENS}-token window.")
print(f"  saved -> {utils.save_table('caption_token_stats', token_stats)}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(tok_len, bins=40, color="#4C72B0")
ax.axvline(cfg.CLIP_MAX_TOKENS, color="crimson", ls="--", label=f"CLIP limit = {cfg.CLIP_MAX_TOKENS}")
ax.axvline(tok_len.mean(), color="#55A868", ls="-", label=f"mean = {tok_len.mean():.0f}")
ax.axvline(p95, color="#DD8452", ls=":", label=f"p95 = {p95:.0f}")
ax.set_title("CLIP token count per cleaned caption")
ax.set_xlabel("tokens")
ax.legend()
plt.tight_layout()
print(f"saved -> {utils.save_fig(fig, 'caption_token_lengths')}")
plt.show()


In [ ]:
clip_text_emb, t_clip_joint = storage.load_or_encode(
    ["clip_text_joint"],
    lambda: clip_embeddings.encode_text(captions, clip_tok, clip_model),
    n_rows=len(captions), cache_usable=CACHE_USABLE, reuse_log=reuse_log,
)
print()
print(f"CLIP joint-space embeddings : {clip_text_emb.shape}  dtype={clip_text_emb.dtype}")
print(f"Time: {t_clip_joint:.1f}s" if t_clip_joint else "Loaded from cache")


In [ ]:
# The Stable Diffusion v1.5 conditioning encoder.
sd_tok, sd_text = clip_embeddings.load_sd_text_encoder()
print(f"Loaded {cfg.CLIP_SD_MODEL}")
for k, v in clip_embeddings.describe(sd_text).items():
    print(f"  {k:<22}{v:,.0f}" if isinstance(v, float) else f"  {k:<22}{v}")

# The pooled vectors are small and always saved; the 77x768 sequence is saved only on request, because
# it is ~710 MB at this subset size and the encoder reproduces it deterministically in under a minute.
clip_seq_emb, clip_sd_pooled, t_clip_seq = storage.load_or_encode(
    ["clip_sd_sequence", "clip_sd_pooled"],
    lambda: clip_embeddings.encode_sequence(captions, sd_tok, sd_text),
    n_rows=len(captions), cache_usable=CACHE_USABLE,
    save=cfg.SAVE_SEQ_EMBEDDINGS, reuse_log=reuse_log,
)
if not cfg.SAVE_SEQ_EMBEDDINGS:
    storage.save_embedding("clip_sd_pooled", clip_sd_pooled)   # the small one is always persisted
    reuse_log["clip_sd_pooled"] = "computed"

print()
print(f"CLIP sequence embeddings : {clip_seq_emb.shape}  dtype={clip_seq_emb.dtype}  "
      f"({clip_seq_emb.nbytes / 1024**2:.0f} MB in memory)")
print(f"CLIP pooled embeddings   : {clip_sd_pooled.shape}  dtype={clip_sd_pooled.dtype}")
print(f"Sequence tensors written to disk: {cfg.SAVE_SEQ_EMBEDDINGS}")
print(f"Time: {t_clip_seq:.1f}s" if t_clip_seq else "Loaded from cache")


In [ ]:
# Because the 77x768 tensors are recomputed rather than stored, confirm the recomputation is in fact
# reproducible: encode the first batch again, with the same seed and the same batch size, and compare.
utils.set_seed()
recomputed, _ = clip_embeddings.encode_sequence(captions[:cfg.BATCH_SIZE], sd_tok, sd_text)

head = clip_seq_emb[:cfg.BATCH_SIZE]
max_delta = float(np.abs(recomputed.astype(np.float32) - head.astype(np.float32)).max())
identical = bool(np.array_equal(recomputed, head))

print(f"Re-encoded the first {cfg.BATCH_SIZE} captions with the same seed and batch size")
print(f"  bit-identical to the first pass : {identical}")
print(f"  max absolute difference         : {max_delta:.2e}")
assert max_delta < 1e-3, "Re-encoding did not reproduce the original tensors"
print()
print("Reproducible: the conditioning tensors can be regenerated on demand instead of stored.")


---
## 8. Text Embeddings — BERT

→ `src/modules/bert_embeddings.py`

**BERT** (`bert-base-uncased`) is our comparison encoder: the same Transformer architecture class and
roughly the same size as CLIP's text tower, but trained with **masked language modelling on text alone**.
It has never seen an image.

That difference is the point. Both encoders produce 768-d sentence vectors that look equally reasonable
on paper, so the only way to tell them apart is to measure them — which is what section 10 does. The
expectation to test: CLIP should separate captions by *visual* content, BERT by *linguistic* similarity,
and only CLIP's space should line up with images at all.

**Why encode it now, in the same pass as CLIP.** The captions are already loaded, cleaned and curated at
this point, so running BERT over them here costs one extra pass over 6,000 short sentences. Leaving it
for later would mean rebuilding the whole subset just to feed a second encoder.

**Pooling.** BERT ships a `pooler_output`, but it is the tanh-projected `[CLS]` state trained for
next-sentence prediction, not for sentence similarity — using it as a sentence embedding is a
well-documented trap. `bert_embeddings.encode_text` computes two honest alternatives instead: the raw
`[CLS]` state, and the attention-mask-weighted **mean** over all real tokens, which is the one to prefer
when the two disagree.

This file is the template for adding another encoder: write one like it, add a call here, and nothing
else in the pipeline changes.


In [ ]:
bert_tok, bert_model = bert_embeddings.load()
print(f"Loaded {cfg.BERT_MODEL}")
for k, v in bert_embeddings.describe(bert_model).items():
    print(f"  {k:<22}{v:,.0f}" if isinstance(v, float) else f"  {k:<22}{v:,}")

bert_cls_emb, bert_mean_emb, t_bert = storage.load_or_encode(
    ["bert_cls", "bert_mean"],
    lambda: bert_embeddings.encode_text(captions, bert_tok, bert_model),
    n_rows=len(captions), cache_usable=CACHE_USABLE, reuse_log=reuse_log,
)
print()
print(f"BERT [CLS] embeddings : {bert_cls_emb.shape}  dtype={bert_cls_emb.dtype}")
print(f"BERT mean embeddings  : {bert_mean_emb.shape}  dtype={bert_mean_emb.dtype}")
print(f"Time: {t_bert:.1f}s" if t_bert else "Loaded from cache")


---
## 9. Storage

→ `src/modules/storage.py`

Embeddings are persisted **alongside the image–caption pairs they came from**, so no later stage ever has
to re-run an encoder or re-derive which caption produced which vector.

```
data/processed/
├── pairs.parquet            # the curated manifest (pair_idx, image_id, caption, paths, ...)
├── pairs.csv                # the same, human-readable
├── metadata.json            # provenance: models, config, shapes, versions, timings
├── images_512/              # 512x512 RGB JPEG cache (one file per distinct image)
└── embeddings/
    ├── clip_text_joint.npy  # (N, 512)  float32   shared text/image space
    ├── clip_sd_pooled.npy   # (N, 768)  float32   SD v1.5 text encoder, pooled
    ├── bert_cls.npy         # (N, 768)  float32   [CLS] token
    └── bert_mean.npy        # (N, 768)  float32   mean-pooled
```

**The contract**: row *i* of every `.npy` is the caption at `pair_idx == i` in `pairs.parquet`. One join
key, enforced by an assertion before anything is written and verified by a round-trip read afterwards.

Figures and summary tables go somewhere separate, under `outputs/`, because they describe a *run* rather
than the dataset itself. Every path comes from `config.py`, so moving the artefacts is a one-line change.

`clip_sd_sequence.npy` — the `(N, 77, 768)` UNet conditioning tensors — is **deliberately not written**:
~710 MB for something the encoder reproduces deterministically in under a minute (verified in section 7).
Flip `SAVE_SEQ_EMBEDDINGS` in `config.py` to trade the disk for the time.


In [ ]:
manifest = storage.write_manifest(subset)

print(f"Wrote the manifest: {len(manifest):,} rows")
print(f"  {utils.rel(cfg.MANIFEST_PARQUET)}")
print(f"  {utils.rel(cfg.MANIFEST_CSV)}")
manifest.head(4)


In [ ]:
# Embedding arrays. Every array is row-checked against the manifest before it is written.
arrays = {
    "clip_text_joint": clip_text_emb,
    "clip_sd_pooled": clip_sd_pooled,
    "bert_cls": bert_cls_emb,
    "bert_mean": bert_mean_emb,
}
if cfg.SAVE_SEQ_EMBEDDINGS:
    arrays["clip_sd_sequence"] = clip_seq_emb

print(f"{'ARRAY':<22}{'SHAPE':<22}{'DTYPE':<10}{'SIZE':>10}")
print("-" * 64)
for name, shape, dtype, mb in storage.save_arrays(arrays, manifest):
    print(f"{name:<22}{str(shape):<22}{dtype:<10}{mb:>8.1f} MB")
print("-" * 64)
print(f"{'total':<54}{sum(a.nbytes for a in arrays.values()) / 1024**2:>8.1f} MB")

# Report -- but never delete -- arrays left over from an earlier run with a different subset size.
stale = storage.find_stale(arrays, len(manifest))
if stale:
    print()
    print("STALE FILES from an earlier run with a different subset (left in place, safe to delete):")
    for path, shape, mb in stale:
        print(f"  {path}  {shape}  {mb:,.0f} MB")


In [ ]:
# Provenance, so a later run -- or a reader of the report -- knows exactly what produced these files.
metadata = storage.build_metadata(
    manifest=manifest,
    arrays=arrays,
    dataset_counts={
        "n_candidate_pairs": int(len(pairs_all)),
        "n_after_cleaning": int(len(pairs_clean)),
    },
    cleaning_steps=prep.CLEANING_STEPS,
    timings={"clip_joint": t_clip_joint, "clip_sequence": t_clip_seq, "bert": t_bert},
    fingerprint=FINGERPRINT,
    reuse_log=reuse_log,
    not_persisted=({} if cfg.SAVE_SEQ_EMBEDDINGS else {
        "clip_sd_sequence": {
            "shape": list(clip_seq_emb.shape), "dtype": str(clip_seq_emb.dtype),
            "reason": "~710 MB; deterministically regenerated by section 7 in under a minute",
        }
    }),
)
storage.save_metadata(metadata)

print(json.dumps(metadata, indent=2)[:1400])
print("...")


In [ ]:
# Round-trip: reload everything from disk and confirm it matches what is in memory. Without this, a
# silent write failure would only surface in a later notebook.
rt = storage.verify_round_trip(manifest, arrays)
probe_img = Image.open(cfg.PROJECT_ROOT / rt["probe_image"])

print("Round-trip verified.")
print(f"  manifest rows : {rt['manifest_rows']:,}")
print(f"  arrays checked: {', '.join(rt['arrays_checked'])}")
print(f"  row 7 caption : {rt['probe_caption']!r}")
print(f"  row 7 image   : {rt['probe_image']} -> {probe_img.size} {probe_img.mode}")


---
## 10. Sanity Checks

→ `src/modules/sanity_checks.py`

Matching shapes only prove the arrays are **well-formed**, not that they are **meaningful** — a bug that
encoded the same caption 6,000 times would pass a shape check. Four checks, cheapest first:

1. **Shape and numerical health** — expected dimensions, no `NaN`/`Inf`, sane vector norms.
2. **Text → image retrieval** — the real test. Encode images with CLIP's vision tower and check that each
   caption retrieves *its own* image out of 500. Chance is 1/500 = 0.2%.
3. **Sibling-caption agreement** — the two captions of one image describe the same scene, so they should
   land closer together than two random captions. Measurable for **every** encoder, BERT included, which
   is what makes the CLIP-vs-BERT comparison possible without an image tower.
4. **Nearest-neighbour inspection** — the qualitative read: what does each encoder *think* is similar?

**A note on retrieval with two captions per image.** Retrieval needs exactly one correct answer per
query, and our subset has two captions pointing at each image. Check 2 therefore runs over the **first
caption of each image only** (`cap_rank == 0`), so no two queries compete for the same target.

The module returns numbers; the plotting stays here, where the figures appear.


In [ ]:
# --- Check 1: shapes and numerical health -------------------------------
expected = sanity_checks.expected_shapes(len(manifest))
health, all_ok = sanity_checks.embedding_health(arrays, expected)

print(f"{'ARRAY':<22}{'SHAPE':<22}{'EXPECTED':<22}{'NaN/Inf':<10}{'MEAN NORM':>10}")
print("-" * 86)
for name, h in health.items():
    ok = h["shape_ok"] and h["all_finite"]
    print(f"{name:<22}{str(tuple(h['shape'])):<22}{str(tuple(h['expected'])):<22}"
          f"{'none' if h['all_finite'] else 'FOUND':<10}{h['norm_mean']:>10.2f}"
          f"{'' if ok else '  <-- FAIL'}")

print("-" * 86)
print("All shapes match the manifest and all values are finite." if all_ok else "CHECK FAILED")
print(f"saved -> {utils.save_table('embedding_health', health)}")
assert all_ok


In [ ]:
# --- Check 2a: encode the retrieval image set with CLIP's vision tower ---
first_caps = manifest[manifest["cap_rank"] == 0].reset_index(drop=True)
align_rows = first_caps.head(min(cfg.N_ALIGN_IMAGES, len(first_caps)))
align_idx = align_rows["pair_idx"].to_numpy()
align_paths = [str(cfg.PROJECT_ROOT / p) for p in align_rows["image_512_path"]]
print(f"Retrieval set: {len(align_idx)} captions over {len(set(align_paths))} distinct images")

# Cached too -- encoding 500 images is the slowest check here.
clip_image_emb, t_align = storage.load_or_encode(
    ["clip_image_align"],
    lambda: clip_embeddings.encode_images(align_paths, clip_proc, clip_model),
    n_rows=len(align_paths), cache_usable=CACHE_USABLE, reuse_log=reuse_log,
)
print()
print(f"CLIP image embeddings: {clip_image_emb.shape}")


In [ ]:
# --- Check 2b: does each caption retrieve its own image? ----------------
ret = sanity_checks.retrieval(clip_text_emb[align_idx], clip_image_emb)

print(f"Text -> image retrieval over {ret['n_queries']} pairs")
print(f"  Recall@1  : {ret['recall_at_1']:6.1%}   (chance {ret['chance']:.2%}  ->  "
      f"{ret['times_better_than_chance']:,.0f}x better)")
print(f"  Recall@5  : {ret['recall_at_5']:6.1%}")
print(f"  Recall@10 : {ret['recall_at_10']:6.1%}")
print(f"  median rank of the correct image : {ret['median_rank_of_correct_image']} of {ret['n_queries']}")
print()
print(f"  cosine, matched pairs    : {ret['cosine_matched_mean']:.4f} +/- {ret['cosine_matched_std']:.4f}")
print(f"  cosine, mismatched pairs : {ret['cosine_mismatched_mean']:.4f} +/- {ret['cosine_mismatched_std']:.4f}")
print(f"  separation               : {ret['separation']:+.4f}")

retrieval_table = {k: v for k, v in ret.items()
                   if k not in ("sim", "ranks", "matched", "mismatched")}
print(f"  saved -> {utils.save_table('clip_retrieval_metrics', retrieval_table)}")

assert ret["recall_at_1"] > 10 * ret["chance"], "Retrieval is near chance -- the embeddings are not aligned"
assert ret["separation"] > 0, "Matched pairs should be more similar than mismatched pairs"
print()
print("PASS -- captions retrieve their own images far above chance.")


In [ ]:
sim, matched, mismatched = ret["sim"], ret["matched"], ret["mismatched"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(mismatched, bins=60, alpha=0.65, label="mismatched pairs", color="#C44E52", density=True)
axes[0].hist(matched, bins=30, alpha=0.80, label="matched pairs", color="#55A868", density=True)
axes[0].axvline(mismatched.mean(), color="#C44E52", ls="--")
axes[0].axvline(matched.mean(), color="#55A868", ls="--")
axes[0].set_title("CLIP cosine similarity: matched vs mismatched")
axes[0].set_xlabel("cosine similarity")
axes[0].legend()

k = 25
im = axes[1].imshow(sim[:k, :k], cmap="viridis")
axes[1].set_title(f"Similarity matrix, first {k} pairs\nbright diagonal = correct alignment")
axes[1].set_xlabel("image")
axes[1].set_ylabel("caption")
fig.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
print(f"saved -> {utils.save_fig(fig, 'clip_matched_vs_mismatched')}")
plt.show()


In [ ]:
# The qualitative view of the same check: a caption and the three images CLIP ranks highest for it.
rng = np.random.default_rng(cfg.SEED)
n_show = 3
picks = rng.choice(ret["n_queries"], n_show, replace=False)
ranks = ret["ranks"]

fig, axes = plt.subplots(n_show, 4, figsize=(13, 3.2 * n_show))
for r, qi in enumerate(picks):
    axes[r, 0].imshow(Image.open(align_paths[qi]))
    axes[r, 0].set_title("ground truth", fontsize=9, color="#55A868")
    axes[r, 0].axis("off")
    for c, ci in enumerate(ranks[qi][:3]):
        axes[r, c + 1].imshow(Image.open(align_paths[ci]))
        axes[r, c + 1].set_title(f"top-{c + 1}  sim={sim[qi, ci]:.3f}{'  correct' if ci == qi else ''}",
                                 fontsize=9, color="#55A868" if ci == qi else "#333333")
        axes[r, c + 1].axis("off")

fig.suptitle("Text -> image retrieval: query caption vs CLIP's top-3 images", y=1.0)
plt.tight_layout()
print(f"saved -> {utils.save_fig(fig, 'clip_retrieval_top3')}")
plt.show()

for r, qi in enumerate(picks):
    print(f"query {r + 1}: {manifest.iloc[align_idx[qi]]['caption']!r}")


In [ ]:
# --- Check 3: do the two captions of one image land near each other? ----
# This is the check BERT can take part in: no image tower required, only the knowledge that two rows
# describe the same scene.
pair_a = manifest[manifest["cap_rank"] == 0].sort_values("image_id")["pair_idx"].to_numpy()
pair_b = manifest[manifest["cap_rank"] == 1].sort_values("image_id")["pair_idx"].to_numpy()
assert len(pair_a) == len(pair_b)

encoders = {
    "CLIP joint": clip_text_emb,
    "CLIP SD pooled": clip_sd_pooled,
    "BERT [CLS]": bert_cls_emb,
    "BERT mean": bert_mean_emb,
}
sibling = sanity_checks.sibling_gap(encoders, pair_a, pair_b, rng)

print(f"Sibling captions (same image) vs random caption pairs, over {len(pair_a):,} images")
print()
print(f"{'ENCODER':<20}{'SIBLING COS':>13}{'RANDOM COS':>13}{'GAP':>9}")
print("-" * 55)
for label, s in sibling.items():
    print(f"{label:<20}{s['sibling_cos']:>13.4f}{s['random_cos']:>13.4f}{s['gap']:>+9.4f}")

best = max(sibling.items(), key=lambda kv: kv[1]["gap"])
print()
print(f"Largest gap: {best[0]} ({best[1]['gap']:+.4f})")
print(f"saved -> {utils.save_table('encoder_sibling_scores', sibling)}")
print("A larger gap means the encoder places two descriptions of one scene closer together than two")
print("unrelated captions -- it is encoding meaning rather than surface word overlap. A high sibling")
print("cosine with an equally high random cosine means the opposite: everything is mapped into a")
print("narrow cone, so the rankings carry little information. Watch for that on the [CLS] row.")


In [ ]:
# The same comparison as a chart. The bar pair matters more than either bar alone -- it is the gap
# that says whether the encoder separates meaning.
labels = list(sibling)
sib_means = [sibling[k]["sibling_cos"] for k in labels]
rnd_means = [sibling[k]["random_cos"] for k in labels]
gaps = [sibling[k]["gap"] for k in labels]
x = np.arange(len(labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
axes[0].bar(x - 0.2, sib_means, 0.4, label="same image", color="#55A868")
axes[0].bar(x + 0.2, rnd_means, 0.4, label="random pair", color="#C44E52")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=20, ha="right")
axes[0].set_ylabel("mean cosine")
axes[0].set_title("Caption pairs: same image vs random")
axes[0].legend()

axes[1].bar(x, gaps, 0.5, color=["#4C72B0" if g < max(gaps) else "#DD8452" for g in gaps])
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=20, ha="right")
axes[1].set_ylabel("gap")
axes[1].set_title("Separation (same image - random), larger is better")

plt.tight_layout()
print(f"saved -> {utils.save_fig(fig, 'encoder_sibling_gap')}")
plt.show()


In [ ]:
# --- Check 4: nearest-neighbour captions, per encoder -------------------
query_idx = int(align_idx[picks[0]])
print(f"QUERY: {manifest.iloc[query_idx]['caption']!r}")
print()
for label, emb in encoders.items():
    idx, sims = sanity_checks.top_neighbours(emb, query_idx, k=5)
    print(label)
    for j, s in zip(idx, sims):
        print(f"   {s:.3f}  {manifest.iloc[int(j)]['caption']}")
    print()


---
## 11. Prompt → Image Test Runs

This is the end-to-end test of everything above, so it stays in the notebook rather than moving into a
module: five **cleaned captions from the curated subset** are fed to **Stable Diffusion v1.5** as prompts,
and each generated image is placed next to the real COCO photo that caption describes.

Using our own cleaned captions rather than invented prompts is deliberate — it tests the pipeline we
actually built, and it gives every generated image a **real photograph as a reference**, which is what
makes the CLIP-score comparison at the end meaningful.

The pipeline's text encoder is the same `clip-vit-large-patch14` whose embeddings we extracted in
section 7, so this run and those arrays are conditioned on the same vectors.

Generation settings come from `config.py` — `N_STEPS`, `GUIDANCE`, `IMAGE_SIZE`, one seeded generator per
prompt so the images are reproducible. Sweeping steps, guidance and schedulers is milestone 2, which is
exactly why those knobs now live in a config file instead of in a cell.

> First run downloads ~4 GB of weights. Set `RUN_SD_GENERATION = False` in `config.py` to skip.


In [ ]:
sd_pipe, SD_REPO = None, None
if cfg.RUN_SD_GENERATION:
    from diffusers import StableDiffusionPipeline

    for repo in cfg.SD_REPO_CANDIDATES:
        try:
            print(f"Loading {repo} ...")
            try:
                sd_pipe = StableDiffusionPipeline.from_pretrained(
                    repo, dtype=cfg.SD_DTYPE, safety_checker=None, requires_safety_checker=False)
            except TypeError:                     # older diffusers spell it torch_dtype
                sd_pipe = StableDiffusionPipeline.from_pretrained(
                    repo, torch_dtype=cfg.SD_DTYPE, safety_checker=None, requires_safety_checker=False)
            SD_REPO = repo
            break
        except Exception as exc:                  # noqa: BLE001 - try the next mirror
            print(f"  unavailable ({type(exc).__name__}: {str(exc)[:120]})")

    if sd_pipe is not None:
        sd_pipe = sd_pipe.to(cfg.DEVICE)
        sd_pipe.set_progress_bar_config(disable=True)
        if cfg.DEVICE.type == "cuda":
            sd_pipe.enable_attention_slicing()    # keeps 512x512 comfortably inside a small VRAM budget
        print()
        print(f"Loaded {SD_REPO} on {cfg.DEVICE}")
        print(f"  text encoder : {type(sd_pipe.text_encoder).__name__} "
              f"(hidden {sd_pipe.text_encoder.config.hidden_size})")
        print(f"  scheduler    : {type(sd_pipe.scheduler).__name__}")
        print(f"  UNet params  : {sum(p.numel() for p in sd_pipe.unet.parameters()) / 1e6:.0f}M")
    else:
        print("\nNo Stable Diffusion checkpoint could be loaded -- skipping generation.")
else:
    print("RUN_SD_GENERATION is False -- skipping generation.")


In [ ]:
# Prompts from the curated subset, long enough to describe a scene rather than a single object.
prompt_rows = (
    manifest[(manifest["n_words"] >= cfg.PROMPT_MIN_WORDS) & (manifest["n_words"] <= cfg.PROMPT_MAX_WORDS)]
    .sample(cfg.N_PROMPTS, random_state=cfg.SEED)
    .reset_index(drop=True)
)

print(f"{cfg.N_PROMPTS} test prompts (cleaned COCO captions from the curated subset):")
print()
for i, row in prompt_rows.iterrows():
    print(f"  {i + 1}. {row['caption']}")
    print(f"     source image: {row['file_name']}")


In [ ]:
results = []

if sd_pipe is not None:
    utils.set_seed()
    for i, row in prompt_rows.iterrows():
        # One generator seeded per prompt: re-running this cell reproduces these exact images.
        generator = torch.Generator(device=cfg.DEVICE).manual_seed(cfg.SEED + i)
        t0 = time.time()
        image = sd_pipe(
            row["caption"], num_inference_steps=cfg.N_STEPS, guidance_scale=cfg.GUIDANCE,
            height=cfg.IMAGE_SIZE, width=cfg.IMAGE_SIZE, generator=generator,
        ).images[0]
        dt = time.time() - t0

        out_path = cfg.GENERATIONS_DIR / f"run_{i + 1}_{row['image_id']}.png"
        image.save(out_path)
        results.append({"prompt": row["caption"], "image": image, "path": out_path,
                        "real": cfg.PROJECT_ROOT / row["image_512_path"], "seconds": dt,
                        "seed": cfg.SEED + i})
        print(f"[{i + 1}/{cfg.N_PROMPTS}] {dt:5.1f}s  seed={cfg.SEED + i}  {row['caption'][:60]}")

    print()
    print(f"Saved {len(results)} images to {utils.rel(cfg.GENERATIONS_DIR)}")
    print(f"Settings: {cfg.N_STEPS} steps, guidance {cfg.GUIDANCE}, {cfg.IMAGE_SIZE}x{cfg.IMAGE_SIZE}")
else:
    print("Generation skipped.")


In [ ]:
if results:
    fig, axes = plt.subplots(2, len(results), figsize=(3.2 * len(results), 7.2))
    for i, r in enumerate(results):
        axes[0, i].imshow(Image.open(r["real"]))
        axes[0, i].set_title("real COCO image", fontsize=9)
        axes[1, i].imshow(r["image"])
        axes[1, i].set_title(f"generated ({r['seconds']:.0f}s)", fontsize=9)
        for row_i in (0, 1):
            axes[row_i, i].axis("off")
        axes[1, i].text(0.5, -0.08, r["prompt"][:58], transform=axes[1, i].transAxes,
                        ha="center", va="top", fontsize=7, wrap=True)

    fig.suptitle("Prompt -> image test runs  (top: real COCO photo, bottom: Stable Diffusion v1.5)",
                 y=0.99)
    plt.tight_layout()
    print(f"saved -> {utils.save_fig(fig, 'prompt_to_image_runs')}")
    plt.show()
else:
    print("No generated images to display.")


In [ ]:
# CLIP cosine similarity for each run: prompt vs generated image, prompt vs real photo, and the two
# images against each other.
if results:
    gen_score, real_score, gen_vs_real = sanity_checks.clip_scores(
        prompts=[r["prompt"] for r in results],
        generated_images=[r["image"] for r in results],
        real_images=[r["real"] for r in results],
        tokenizer=clip_tok, model=clip_model, processor=clip_proc,
    )

    print(f"{'#':<3}{'CLIP(txt,gen)':>14}{'CLIP(txt,real)':>16}{'CLIP(gen,real)':>16}  PROMPT")
    print("-" * 100)
    for i, r in enumerate(results):
        print(f"{i + 1:<3}{gen_score[i]:>14.4f}{real_score[i]:>16.4f}{gen_vs_real[i]:>16.4f}  "
              f"{r['prompt'][:44]}")
    print("-" * 100)
    print(f"{'avg':<3}{gen_score.mean():>14.4f}{real_score.mean():>16.4f}{gen_vs_real.mean():>16.4f}")

    runs_table = {
        "settings": {"model": SD_REPO, "steps": cfg.N_STEPS, "guidance": cfg.GUIDANCE,
                     "size": [cfg.IMAGE_SIZE, cfg.IMAGE_SIZE], "seeds": [r["seed"] for r in results]},
        "runs": [
            {"prompt": r["prompt"], "image": utils.rel(r["path"]),
             "seconds": round(r["seconds"], 1), "seed": r["seed"],
             "clip_text_generated": float(gen_score[i]),
             "clip_text_real": float(real_score[i]),
             "clip_generated_real": float(gen_vs_real[i])}
            for i, r in enumerate(results)
        ],
        "means": {"clip_text_generated": float(gen_score.mean()),
                  "clip_text_real": float(real_score.mean()),
                  "clip_generated_real": float(gen_vs_real.mean())},
    }
    print(f"saved -> {utils.save_table('generation_runs', runs_table)}")

    x = np.arange(len(results))
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.bar(x - 0.2, gen_score, 0.4, label="generated image", color="#4C72B0")
    ax.bar(x + 0.2, real_score, 0.4, label="real COCO photo", color="#55A868")
    ax.axhline(gen_score.mean(), color="#4C72B0", ls="--", lw=1,
               label=f"generated mean = {gen_score.mean():.3f}")
    ax.axhline(real_score.mean(), color="#55A868", ls=":", lw=1,
               label=f"real mean = {real_score.mean():.3f}")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{i + 1}" for i in x])
    ax.set_xlabel("prompt")
    ax.set_ylabel("CLIP cosine with the prompt")
    ax.set_title("How closely each image matches its caption")
    ax.legend(fontsize=8)
    plt.tight_layout()
    print(f"saved -> {utils.save_fig(fig, 'clip_scores_per_prompt')}")
    plt.show()


### Reading the table

Every number is a **cosine similarity in CLIP's joint space**, the shared space where a sentence vector
and an image vector can be compared directly. The scale runs from -1 to 1, but text-image scores sit in
a narrow band in practice: roughly **0.25–0.35** for a caption and an image that genuinely match, and
around 0.1 for an unrelated pair.

| Column | Compares | The question it answers |
|--------|----------|-------------------------|
| `CLIP(txt,gen)` | prompt vs generated image | did the model draw what the prompt asked for? |
| `CLIP(txt,real)` | prompt vs the real COCO photo | how well does that caption match its own photograph? |
| `CLIP(gen,real)` | generated image vs real photo | how close are the two pictures to each other? |

Three things to read off it:

- **`CLIP(txt,gen)` landing inside the 0.25–0.35 band** means the prompt survived the whole pipeline —
  cleaning, tokenising, encoding and conditioning — and reached the UNet intact. That is what this run
  is checking.
- **`CLIP(txt,real)` is the reference column.** It is the score a *real* photograph gets against the same
  caption, so it sets the scale for the first column rather than being a target to beat. Individual rows
  go either way: a generated image can score higher than the real photo, because it renders the
  caption's literal content without the incidental clutter a real scene contains.
- **`CLIP(gen,real)` is on a different scale entirely** and runs much higher, because two images share
  far more structure with each other than a sentence ever shares with an image. Compare it across rows,
  never against the first two columns.

What these numbers do **not** measure is whether the images look real. CLIP score captures agreement
with the prompt, not fidelity or diversity — that is what FID and Inception Score are for, in the
evaluation stage.


---
## 12. Artefacts

The inventory below is what this notebook leaves on disk. The written-up version — what each artefact is
for and which milestone consumes it — lives in **[`docs/milestone_1_artifacts.md`](../../docs/milestone_1_artifacts.md)**,
so the report can be assembled without re-running anything.


In [ ]:
print("ARTEFACTS")
print("-" * 86)
print(f"  {'data/processed/pairs.parquet':<52}{len(manifest):,} rows")
print(f"  {'data/processed/pairs.csv':<52}same, human-readable")
print(f"  {'data/processed/metadata.json':<52}provenance")
print(f"  {'data/processed/images_512/':<52}{manifest['image_id'].nunique():,} x "
      f"{cfg.IMAGE_SIZE}x{cfg.IMAGE_SIZE} RGB ({cache_info['cache_mb']:,.0f} MB)")
for name, arr in arrays.items():
    print(f"  {'data/processed/embeddings/' + name + '.npy':<52}{str(arr.shape):<20}"
          f"{arr.nbytes / 1024**2:>7.1f} MB")
if results:
    print(f"  {'outputs/generations/':<52}{len(results)} PNG test runs")

plots = sorted(cfg.PLOTS_DIR.glob("*.png"))
tables = sorted(cfg.TABLES_DIR.glob("*.json"))
print(f"  {'outputs/plots/':<52}{len(plots)} figures")
for f in plots:
    print(f"      {f.name}")
print(f"  {'outputs/tables/':<52}{len(tables)} summary tables")
for f in tables:
    print(f"      {f.name}")

print()
print("ENCODE TIME  (cached = loaded from disk)")
print("-" * 86)
for label, secs in (("CLIP joint", t_clip_joint), ("CLIP sequence", t_clip_seq), ("BERT", t_bert)):
    print(f"  {label:<24}" + (f"{secs:>7.1f}s" if secs else "  cached"))
print()
print(f"  subset fingerprint : {FINGERPRINT}")
for name, state in reuse_log.items():
    print(f"  {name:<24}{state}")


---
### Where this leads next

Everything above is deliberately **training-free**: pretrained encoders, a pretrained generator, and a
dataset prepared so that the next stages only have to read it.

**Immediately reusable from here**

- `pairs.parquet` — the join key tying every embedding row to its caption and its image.
- `images_512/` — the real-image reference set. Use `prep.normalize_pixels(img, "pm1")` before the SD VAE
  encoder, or `"01"` for FID feature extraction.
- `clip_text_joint.npy` plus `sanity_checks.retrieval` — already an automated CLIP-score evaluator for
  generated images.
- `clip_embeddings.encode_sequence` — regenerates the `77 × 768` UNet conditioning tensors on demand,
  deterministically.

**The open design question, stated now because it shapes the next stage.** A pretrained diffusion model
only understands the embedding space it was trained in: BERT vectors cannot be fed to Stable Diffusion's
UNet as-is, and with a few thousand pairs and a single Kaggle T4, unfreezing the UNet to teach it a new
space is not realistic. Section 10's comparison is what makes that concrete rather than assumed — it
shows *how* the two spaces differ, which is the evidence behind choosing one of:

1. **Compare pretrained models that already use different text encoders** (CLIP-conditioned vs
   T5-conditioned), which needs no training at all and fits the compute budget. Switching the encoder is
   now a new file in `src/modules/` plus one call here, not a rewrite.
2. **Freeze both models and train only a small projection layer** mapping BERT's space into SD's
   conditioning space — far cheaper than unfreezing, with weaker images expected. Explaining *why* they
   are weaker, with section 10's numbers as evidence, is itself a result.

**Then**: guidance-scale and inference-step sweeps, noise-schedule and scheduler comparisons, and FID and
Inception Score against the 3,000-image real set prepared here. Those knobs are already in `config.py`.
